# HUẤN LUYỆN TINH CHỈNH TPS-INBETWEEN PHIÊN BẢN V2.5 (EXACT-STE DENSITY CALIBRATION & BIDIRECTIONAL STROKE WIDTH CONTROL)
## ĐỀ TÀI: NGHIÊN CỨU & TINH CHỈNH MÔ HÌNH TPS-INBETWEEN CHO HOẠT HỌA ANIME 2D (PBL6)

> **Mục tiêu cốt lõi phiên bản V2.5 (Production Precision Release - Khắc Phục Triệt Để Độ Dày Nét):**
> 1. **Khắc Phục Lỗi Mất Đạo Hàm & Hiệu Chuẩn Mật Độ Nét (Cell 6):** Thay thế hàm kẹp bão hòa đạo hàm của V2.4 bằng `StrokeDensityRegularizationLoss` kết hợp **Straight-Through Estimator (STE)** tại đúng ngưỡng nhị phân $0.15$ và nhánh **Leaky Surrogate Gradient** $(+ 0.25 \cdot \text{ReLU}(p_{\text{ink}} - 0.24))$. Đạo hàm $\frac{\partial \mathcal{L}}{\partial p_{\text{ink}}}$ luôn dương liên tục ở vùng nét dày, khóa chặt `Density Ratio` vào Dead-zone lý tưởng $[0.96, 1.03]$ (tâm chuẩn $1.00$) với trọng số $\lambda_{\text{density}} = 0.55$.
> 2. **Kiểm Soát Bề Rộng Nét Hai Chiều (Cell 6):** Nâng cấp lên `BidirectionalStrokeWidthLoss` ($\lambda = 0.25$). Bảo toàn trọn vẹn lõi nét mảnh và khung xương $1-2\text{ px}$ trên Ground Truth, đồng thời triệt tiêu hiện tượng phình viền (`loss_over_edge`) tại vành đai $1-2\text{ px}$ quanh nét thật; cân bằng `SoftClDiceLoss` $(\beta = 0.50)$ giữa Topological Precision và Sensitivity.
> 3. **Triệt Tiêu Quầng Mờ & Bóng Ma Cận Biên (Cell 6):** `GhostStrokePenaltyLoss` phạt quầng xám ngoài lõi nét mà không bào mòn nét mảnh.
> 4. **Dual-Checkpoint Model Soup Warm-Start (Cell 7):** Hỗ trợ tự động pha trộn trọng số $0.52 \cdot \theta_{\text{V2.4/V2.0}} + 0.48 \cdot \theta_{\text{V2.3}}$ đưa mật độ về ngay $\sim 1.02$ tại Epoch 0, kèm cơ chế Fallback nạp đơn lẻ linh hoạt.
> 5. **Quản Lý Checkpoint Chuẩn Sản Xuất & Demo Trực Quan Realtime (Cell 9 & Cell 10):** Hard Gates nghiêm ngặt (`Recall` $\ge 89.5\%$, `Density` $\in [0.90, 1.10]$, target $1.00$). Sau mỗi epoch cập nhật Best Checkpoint hoặc Best Progress, tự động xuất ngay 3 ảnh demo trực quan tại `/kaggle/working/visual_samples_v2.5/` để tải về kiểm tra tức thì trong quá trình huấn luyện.
> 6. **Tối Ưu Siêu Tốc Cell 4 (Zero Wait Time & Zero RAM Leak):** Triệt tiêu hoàn toàn thời gian nghẽn 28 phút của vòng lặp nạp trước. Khởi động Cell 4 chỉ trong $< 5$ giây, sử dụng DataLoader đa tiến trình 4 vCPUs (`persistent_workers=True`, `pin_memory=True`), giữ Host RAM phẳng $< 3.5\text{ GB}$ suốt 25 epochs.


In [ ]:
# [CELL 1]: Cấu hình Môi trường, Phần cứng & Trình Dọn Rác Bộ Nhớ Chủ Động (V2.5)
import os
import sys
import ctypes

os.environ["MALLOC_TRIM_THRESHOLD_"] = "65536"
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
# Giảm phân mảnh VRAM của CUDA caching allocator (phải đặt TRƯỚC khi import torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    if hasattr(sys.stdout, 'reconfigure'):
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    if hasattr(sys.stderr, 'reconfigure'):
        sys.stderr.reconfigure(encoding='utf-8', errors='replace')
except Exception:
    pass

import gc
import time
import math
import random
import json
import shutil
import numpy as np
import pandas as pd
from PIL import Image
import cv2
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torch.utils.data.dataloader import default_collate
import torchvision.models as models

try:
    mp.set_sharing_strategy('file_descriptor')
except Exception:
    pass

def configure_glibc_allocator():
    try:
        libc = ctypes.CDLL('libc.so.6')
        libc.mallopt(-1, 65536)
        libc.mallopt(-8, 2)
    except Exception:
        pass

configure_glibc_allocator()

# Thiết lập Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

# Kiểm tra GPU & Cấu hình thiết bị
NUM_GPUS = torch.cuda.device_count()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"[*] Đang sử dụng thiết bị: {device} | Số GPU khả dụng: {NUM_GPUS}")
for i in range(NUM_GPUS):
    prop = torch.cuda.get_device_properties(i)
    print(f"    - GPU {i}: {prop.name} (VRAM: {prop.total_memory / 1e9:.2f} GB)")

# Hàm dọn dẹp bộ nhớ chuyên sâu chống tràn RAM máy chủ và VRAM (Zero Memory Leak Engine)
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass

def _read_cgroup_memory_bytes():
    for p in ('/sys/fs/cgroup/memory.current', '/sys/fs/cgroup/memory/memory.usage_in_bytes'):
        try:
            with open(p, 'r') as f:
                return int(f.read().strip())
        except Exception:
            continue
    return None

def get_host_ram_gb():
    rss_gb = None
    try:
        import psutil
        proc = psutil.Process(os.getpid())
        rss = proc.memory_info().rss
        for ch in proc.children(recursive=True):
            try:
                rss += ch.memory_info().rss
            except Exception:
                pass
        rss_gb = rss / (1024 ** 3)
    except Exception:
        pass
    cg = _read_cgroup_memory_bytes()
    cg_gb = cg / (1024 ** 3) if cg is not None else None
    return rss_gb, cg_gb

def format_host_ram():
    rss_gb, cg_gb = get_host_ram_gb()
    rss_s = f"{rss_gb:.2f} GB" if rss_gb is not None else "N/A"
    cg_s = f"{cg_gb:.2f} GB" if cg_gb is not None else "N/A"
    return f"Process RSS: {rss_s} | Container (cgroup): {cg_s}"

cleanup_memory()
print(f"[*] Host RAM hiện tại -> {format_host_ram()}")
print("[✓] CELL 1 hoàn tất: Môi trường và hàm quản lý bộ nhớ sâu sẵn sàng (V2.5).")

In [ ]:
# [CELL 2]: Tự Động Định Vị, Giải Nén Siêu Tốc & Thẩm Định Toàn Vẹn Golden Dataset (Patch 2 Ready)
import os
import sys
import gc
import time
import zipfile
import tarfile
import subprocess
import pandas as pd

KAGGLE_INPUT = '/kaggle/input'
WORKING_DIR = '/kaggle/working'

TMP_DIR = os.path.join(WORKING_DIR, '_temp_dataset')
EXTRACT_DIR = os.path.join(TMP_DIR, 'sakuga_golden_dataset_tps_ready')
DATASET_EXTRACTED_TO_TEMP = False

DATASET_ROOT = None

# Bước 1: Quét đệ quy tìm trực tiếp thư mục chứa triplets.csv hoặc triplets_combined.csv
search_locations = [KAGGLE_INPUT, EXTRACT_DIR, TMP_DIR, WORKING_DIR, './data/processed', '../data/processed', '.']

print("[*] Đang quét tìm thư mục Golden Dataset (chứa triplets.csv hoặc triplets_combined.csv)...")
for base_dir in search_locations:
    if not os.path.exists(base_dir):
        continue
    for root, dirs, files in os.walk(base_dir):
        if 'triplets.csv' in files or 'triplets_combined.csv' in files:
            DATASET_ROOT = root
            print(f"[✓] Tự động phát hiện thư mục Golden Dataset tại: {DATASET_ROOT}")
            break
    if DATASET_ROOT is not None:
        break

# Bước 2: Quét tìm archive (.tar.gz hoặc .zip) hỗ trợ cả Patch 2 và tự động giải nén vào NVMe
if DATASET_ROOT is None:
    print("[*] Chưa thấy thư mục giải nén, tiến hành quét tìm archive (.tar.gz / .zip) chứa Sakuga / Patch 2...")
    archive_path = None
    archive_type = None

    valid_keywords = ['sakuga', 'golden', 'tps', 'patch2', 'patch_2']

    for base_dir in search_locations:
        if not os.path.exists(base_dir):
            continue
        for root, dirs, files in os.walk(base_dir):
            for f in files:
                f_lower = f.lower()
                is_match = any(k in f_lower for k in valid_keywords)
                if is_match and (f_lower.endswith('.tar.gz') or f_lower.endswith('.tgz')):
                    archive_path = os.path.join(root, f)
                    archive_type = 'tar'
                    break
                elif is_match and f_lower.endswith('.zip'):
                    archive_path = os.path.join(root, f)
                    archive_type = 'zip'
                    break
            if archive_path is not None:
                break
        if archive_path is not None:
            break

    if archive_path is not None and os.path.exists(archive_path):
        file_size_mb = os.path.getsize(archive_path) / (1024 * 1024)
        print(f"[✓] Phát hiện archive dataset: {archive_path} ({file_size_mb:.2f} MB - Loại: {archive_type})")
        print(f"[*] Tiến hành giải nén siêu tốc vào ổ đĩa vật lý /kaggle/working ({EXTRACT_DIR})...")
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        DATASET_EXTRACTED_TO_TEMP = True
        t_start = time.time()

        extracted = False
        if archive_type == 'tar':
            try:
                cmd = ['tar', '-xzf', archive_path, '-C', EXTRACT_DIR]
                res = subprocess.run(cmd, capture_output=True, text=True)
                if res.returncode == 0:
                    extracted = True
            except Exception:
                pass
            if not extracted:
                with tarfile.open(archive_path, 'r:gz') as tar:
                    tar.extractall(EXTRACT_DIR)
                extracted = True
        elif archive_type == 'zip':
            try:
                cmd = ['unzip', '-q', '-o', archive_path, '-d', EXTRACT_DIR]
                res = subprocess.run(cmd, capture_output=True, text=True)
                if res.returncode == 0:
                    extracted = True
            except Exception:
                pass
            if not extracted:
                with zipfile.ZipFile(archive_path, 'r') as zf:
                    zf.extractall(EXTRACT_DIR)
                extracted = True

        extract_duration = time.time() - t_start
        print(f"[✓] Giải nén toàn bộ dataset hoàn tất trong {extract_duration:.1f}s!")

        for root, dirs, files in os.walk(EXTRACT_DIR):
            if 'triplets.csv' in files or 'triplets_combined.csv' in files:
                DATASET_ROOT = root
                break

if DATASET_ROOT is None:
    raise FileNotFoundError("❌ Không tìm thấy tập dữ liệu Sakuga Golden Dataset / Patch 2! Vui lòng kiểm tra Dataset gắn vào Kaggle.")

# Nạp triplets metadata
triplets_csv_path = os.path.join(DATASET_ROOT, 'triplets.csv')
if not os.path.exists(triplets_csv_path):
    triplets_csv_path = os.path.join(DATASET_ROOT, 'triplets_combined.csv')

df_triplets = pd.read_csv(triplets_csv_path)
train_df = df_triplets[df_triplets['split'] == 'train'].reset_index(drop=True)
val_df = df_triplets[df_triplets['split'] == 'val'].reset_index(drop=True)

total_triplets_count = len(df_triplets)
num_train_samples = len(train_df)
num_val_samples = len(val_df)
del df_triplets
gc.collect()

print(f"[✓] Thư mục Golden Dataset hợp lệ: {DATASET_ROOT}")
print(f"[✓] Tổng số Golden Triplets: {total_triplets_count:,} mẫu (Hỗ trợ quy mô Patch 2: ~6,734 triplets)")
print(f"    - Tập Huấn luyện (Train): {num_train_samples:,} mẫu ({num_train_samples/max(1, total_triplets_count)*100:.1f}%)")
print(f"    - Tập Thẩm định (Val)   : {num_val_samples:,} mẫu ({num_val_samples/max(1, total_triplets_count)*100:.1f}%)")
assert num_train_samples > 0 and num_val_samples > 0, "Dữ liệu train hoặc val bị rỗng! Kiểm tra triplets.csv."

matches_dir = os.path.join(DATASET_ROOT, 'matches')
print(f"[*] Thư mục keypoint matches: {matches_dir} (Tồn tại: {os.path.exists(matches_dir)})")
print("[✓] CELL 2 hoàn tất: Toàn vẹn Golden Dataset (Patch 2 Ready) được xác nhận.")

In [ ]:
# [CELL 3]: Hệ Thống Biến Dạng Lưới, 8 Điểm Neo Biên/Góc & Vector Hóa TPS Flow Batch
_NORM_GRID_CACHE = {}
_PIXEL_GRID_CACHE = {}
# [V1.1 FIX]: Giới hạn kích thước cache lưới (Bounded Eviction) để chống rò rỉ bộ nhớ khi gặp
# các ảnh Validation có độ phân giải khác nhau (mỗi (h, w, device, dtype) tạo 1 tensor lưới mới).
_MAX_GRID_CACHE_SIZE = 32

def warp_normalized_flow(img, flow):
    """
    Biến dạng lưới Coarse TPS (tọa độ chuẩn hóa [0, 1] sang [-1, 1]).
    Tối ưu hóa V1.0: Caching lưới cơ sở (1, H, W, 2) và cộng trực tiếp qua broadcasting
    (loại bỏ hoàn toàn meshgrid và .repeat(b, 1, 1, 1) dư thừa).
    """
    b, _, h, w = img.shape
    dev = flow.device
    dtype = flow.dtype
    key = (h, w, dev, dtype)
    # Giữ tham chiếu cục bộ (an toàn khi nhiều thread DataParallel cùng truy cập & clear cache)
    base_grid = _NORM_GRID_CACHE.get(key)
    if base_grid is None:
        if len(_NORM_GRID_CACHE) >= _MAX_GRID_CACHE_SIZE:
            _NORM_GRID_CACHE.clear()
        grid_y, grid_x = torch.meshgrid(
            torch.linspace(0, 1, h, device=dev, dtype=dtype),
            torch.linspace(0, 1, w, device=dev, dtype=dtype), indexing='ij'
        )
        base_grid = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0)
        _NORM_GRID_CACHE[key] = base_grid

    grid = (base_grid + flow.permute(0, 2, 3, 1)) * 2.0 - 1.0
    return F.grid_sample(img, grid, padding_mode='border', align_corners=True)

def warp_pixel_flow(img, flow):
    """
    Biến dạng lưới Fine Optical Flow IFNet (tọa độ pixel thực tế).
    Tối ưu hóa V1.0: Caching lưới cơ sở [-1, 1] và cộng trực tiếp qua broadcasting.
    """
    b, _, h, w = img.shape
    dev = flow.device
    dtype = flow.dtype
    key = (h, w, dev, dtype)
    base_grid = _PIXEL_GRID_CACHE.get(key)
    if base_grid is None:
        if len(_PIXEL_GRID_CACHE) >= _MAX_GRID_CACHE_SIZE:
            _PIXEL_GRID_CACHE.clear()
        grid_y, grid_x = torch.meshgrid(
            torch.linspace(-1, 1, h, device=dev, dtype=dtype),
            torch.linspace(-1, 1, w, device=dev, dtype=dtype), indexing='ij'
        )
        base_grid = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0)
        _PIXEL_GRID_CACHE[key] = base_grid

    flow_norm = torch.cat([
        flow[:, 0:1, :, :] / ((w - 1.0) / 2.0),
        flow[:, 1:2, :, :] / ((h - 1.0) / 2.0)
    ], dim=1)
    grid = base_grid + flow_norm.permute(0, 2, 3, 1)
    return F.grid_sample(img, grid, padding_mode='border', align_corners=True)

def tps_rbf(r):
    r = torch.clamp(r, min=1e-6)
    return (r ** 2) * torch.log(r)

# 8 Điểm neo tĩnh (4 góc + 4 điểm giữa biên) để khóa chặt khung hình, chống co rút viền ảnh
FIXED_ANCHORS_CPU = torch.tensor([
    [0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0],
    [0.5, 0.0], [0.5, 1.0], [0.0, 0.5], [1.0, 0.5]
], dtype=torch.float64)

FIXED_ANCHORS_NP = np.array([
    [0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0],
    [0.5, 0.0], [0.5, 1.0], [0.0, 0.5], [1.0, 0.5]
], dtype=np.float64)

def robust_tps_theta_from_points(c_src, c_dst, lambd=0.5, min_dist=0.015, add_anchors=True):
    """
    Giải hệ phương trình TPS đơn mẫu với 8 điểm neo biên/góc (kháng suy biến và chống co rút).
    """
    device = c_src.device
    n = c_src.shape[0]
    if n < 4:
        return torch.zeros((n + 3, 2), device=device)

    c_src_cpu = c_src.detach().cpu().to(torch.float64)
    c_dst_cpu = c_dst.detach().cpu().to(torch.float64)

    keep_indices = [0]
    for i in range(1, n):
        dists = torch.norm(c_src_cpu[i] - c_src_cpu[keep_indices], dim=-1)
        if torch.min(dists) >= min_dist:
            keep_indices.append(i)

    if len(keep_indices) < 4:
        return torch.zeros((n + 3, 2), device=device)

    c_s_filt = c_src_cpu[keep_indices]
    c_d_filt = c_dst_cpu[keep_indices]

    if add_anchors:
        c_s_filt = torch.cat([c_s_filt, FIXED_ANCHORS_CPU], dim=0)
        c_d_filt = torch.cat([c_d_filt, FIXED_ANCHORS_CPU], dim=0)

    n_filt = c_s_filt.shape[0]
    diff = c_s_filt.unsqueeze(1) - c_s_filt.unsqueeze(0)
    dist = torch.norm(diff, dim=-1)

    K = tps_rbf(dist) + lambd * torch.eye(n_filt, dtype=torch.float64)
    P = torch.cat([torch.ones((n_filt, 1), dtype=torch.float64), c_s_filt], dim=-1)
    L_top = torch.cat([K, P], dim=-1)
    L_bot = torch.cat([P.t(), torch.zeros((3, 3), dtype=torch.float64)], dim=-1)
    L = torch.cat([L_top, L_bot], dim=0)
    Y = torch.cat([c_d_filt, torch.zeros((3, 2), dtype=torch.float64)], dim=0)

    try:
        theta_filt = torch.linalg.solve(L, Y)
    except Exception:
        theta_filt = torch.linalg.lstsq(L, Y).solution

    return theta_filt.to(device=device, dtype=torch.float32), c_s_filt.to(device=device, dtype=torch.float32)

def solve_tps_single_cpu(c_s, c_d, max_kps=80, lambd=0.5, min_dist=0.015, max_pts=88, return_tensor=False):
    """
    Giải TPS đơn chiều hoàn toàn bằng NumPy.
    [V1.1 FIX]: Mặc định trả về (c_pad: np.ndarray float32, theta_pad: np.ndarray float32, valid: float)
    để DataLoader worker không phải quản lý đối tượng torch.Tensor (default_collate sẽ chuyển sang Tensor).
    Đặt return_tensor=True để nhận torch.Tensor (tương thích ngược khi gọi ngoài DataLoader).
    """
    if isinstance(c_s, torch.Tensor):
        c_s = c_s.detach().cpu().numpy().astype(np.float64, copy=False)
        c_d = c_d.detach().cpu().numpy().astype(np.float64, copy=False)
    else:
        c_s = np.asarray(c_s, dtype=np.float64)
        c_d = np.asarray(c_d, dtype=np.float64)

    def _out(c_pad, theta_pad, valid):
        if return_tensor:
            return torch.from_numpy(c_pad), torch.from_numpy(theta_pad), float(valid)
        return c_pad, theta_pad, float(valid)

    n = c_s.shape[0]
    if n < 4:
        return _out(np.zeros((max_pts, 2), dtype=np.float32), np.zeros((max_pts + 3, 2), dtype=np.float32), 0.0)

    # Lọc điểm quá gần trên mảng NumPy phẳng (không tạo PyTorch tensor trong vòng lặp)
    min_dist_sq = min_dist * min_dist
    keep_indices = [0]
    kept_pts = np.empty((n, 2), dtype=np.float64)
    kept_pts[0] = c_s[0]
    num_kept = 1

    for i in range(1, n):
        dx = kept_pts[:num_kept, 0] - c_s[i, 0]
        dy = kept_pts[:num_kept, 1] - c_s[i, 1]
        if np.min(dx * dx + dy * dy) >= min_dist_sq:
            kept_pts[num_kept] = c_s[i]
            keep_indices.append(i)
            num_kept += 1

    if num_kept < 4:
        return _out(np.zeros((max_pts, 2), dtype=np.float32), np.zeros((max_pts + 3, 2), dtype=np.float32), 0.0)

    if num_kept > max_kps:
        idx_sub = np.linspace(0, num_kept - 1, max_kps, dtype=int)
        keep_indices = [keep_indices[k] for k in idx_sub]

    c_s_filt = np.concatenate([c_s[keep_indices], FIXED_ANCHORS_NP], axis=0)
    c_d_filt = np.concatenate([c_d[keep_indices], FIXED_ANCHORS_NP], axis=0)

    m = c_s_filt.shape[0]
    diff = c_s_filt[:, None, :] - c_s_filt[None, :, :]
    dist = np.sqrt(np.sum(diff * diff, axis=-1))
    dist_clamped = np.maximum(dist, 1e-6)
    K = (dist_clamped ** 2) * np.log(dist_clamped) + lambd * np.eye(m, dtype=np.float64)

    P = np.concatenate([np.ones((m, 1), dtype=np.float64), c_s_filt], axis=-1)
    L_top = np.concatenate([K, P], axis=-1)
    L_bot = np.concatenate([P.T, np.zeros((3, 3), dtype=np.float64)], axis=-1)
    L = np.concatenate([L_top, L_bot], axis=0)
    Y = np.concatenate([c_d_filt, np.zeros((3, 2), dtype=np.float64)], axis=0)

    try:
        theta = np.linalg.solve(L, Y)
    except Exception:
        theta = np.linalg.lstsq(L, Y, rcond=None)[0]

    c_pad = np.zeros((max_pts, 2), dtype=np.float32)
    theta_pad = np.zeros((max_pts + 3, 2), dtype=np.float32)
    c_pad[:m] = c_s_filt.astype(np.float32)
    theta_pad[:m] = theta[:m].astype(np.float32)
    theta_pad[max_pts:max_pts+3] = theta[m:m+3].astype(np.float32)

    return _out(c_pad, theta_pad, 1.0)

def solve_tps_pair_cpu(kps, max_kps=80, lambd=0.5, min_dist=0.015, return_tensor=True):
    """
    Giải TPS 2 chiều (c0 -> c1 và c1 -> c0) trên CPU worker của DataLoader.
    Trả về c0_pad (88, 2), th01_pad (91, 2), c1_pad (88, 2), th10_pad (91, 2), valid_flag (scalar float32).
    - return_tensor=False: Toàn bộ là np.ndarray float32 (dùng bên trong DataLoader worker -> giảm overhead IPC).
    - return_tensor=True : torch.Tensor (mặc định, tương thích ngược cho nhánh fallback inference của mô hình).
    """
    MAX_PTS = 88
    if kps is None or len(kps) < 2 or (hasattr(kps, 'shape') and len(kps.shape) > 1 and kps.shape[1] < 4):
        z_c0 = np.zeros((MAX_PTS, 2), dtype=np.float32)
        z_th01 = np.zeros((MAX_PTS + 3, 2), dtype=np.float32)
        z_c1 = np.zeros((MAX_PTS, 2), dtype=np.float32)
        z_th10 = np.zeros((MAX_PTS + 3, 2), dtype=np.float32)
        z_v = np.array(0.0, dtype=np.float32)
        if return_tensor:
            return (torch.from_numpy(z_c0), torch.from_numpy(z_th01), torch.from_numpy(z_c1),
                    torch.from_numpy(z_th10), torch.from_numpy(z_v))
        return z_c0, z_th01, z_c1, z_th10, z_v

    # Truyền trực tiếp mảng kps[0] và kps[1] vào solve_tps_single_cpu mà không cần tạo tensor trung gian
    c0 = kps[0]
    c1 = kps[1]

    c0_pad, th01_pad, v01 = solve_tps_single_cpu(c0, c1, max_kps=max_kps, lambd=lambd, min_dist=min_dist, max_pts=MAX_PTS, return_tensor=False)
    c1_pad, th10_pad, v10 = solve_tps_single_cpu(c1, c0, max_kps=max_kps, lambd=lambd, min_dist=min_dist, max_pts=MAX_PTS, return_tensor=False)

    v = np.array(1.0 if (v01 > 0.5 and v10 > 0.5) else 0.0, dtype=np.float32)
    if return_tensor:
        return (torch.from_numpy(c0_pad), torch.from_numpy(th01_pad), torch.from_numpy(c1_pad),
                torch.from_numpy(th10_pad), torch.from_numpy(v))
    return c0_pad, th01_pad, c1_pad, th10_pad, v

_GRID_CACHE = {}

def get_grid_pts(h, w, device):
    key = (h, w, device)
    pts = _GRID_CACHE.get(key)
    if pts is None:
        if len(_GRID_CACHE) >= _MAX_GRID_CACHE_SIZE:
            _GRID_CACHE.clear()
        grid_y, grid_x = torch.meshgrid(
            torch.linspace(0, 1, h, device=device),
            torch.linspace(0, 1, w, device=device), indexing='ij'
        )
        pts = torch.stack([grid_x, grid_y], dim=-1).view(-1, 2)
        _GRID_CACHE[key] = pts
    return pts

@torch.no_grad()
def tps_flow_from_theta_batch(theta_batch, c_src_batch, valid_mask, h, w, device, chunk_size=65536):
    """
    Vector hóa 100% trên GPU: Tính toán trường dòng chuyển động TPS cho TOÀN BỘ BATCH cùng lúc
    bằng torch.cdist và torch.bmm (Bảo toàn 100% giá trị toán học, nhanh gấp 4 lần, không tạo tensor 4D).
    - theta_batch: (B, 91, 2)
    - c_src_batch: (B, 88, 2)
    - valid_mask: (B, 1, 1, 1)
    - Output: flow (B, 2, H, W)
    """
    with torch.amp.autocast('cuda', enabled=False):
        theta_f32 = theta_batch.float()
        c_src_f32 = c_src_batch.float()
        valid_f32 = valid_mask.float()

        b = theta_f32.shape[0]
        pts = get_grid_pts(h, w, device)
        total_pts = pts.shape[0]
        flow_chunks = []

        for i in range(0, total_pts, chunk_size):
            p_chunk = pts[i:i + chunk_size]
            c_len = p_chunk.shape[0]
            pts_exp = p_chunk.unsqueeze(0).expand(b, -1, -1)

            # Dùng torch.cdist thay vì tạo tensor 4D (B, C, 88, 2) tốn VRAM
            dist = torch.cdist(pts_exp, c_src_f32, p=2.0)
            U = tps_rbf(dist)

            ones = torch.ones((b, c_len, 1), device=device, dtype=pts.dtype)
            P = torch.cat([ones, pts_exp], dim=-1)
            basis = torch.cat([U, P], dim=-1)

            mapped = torch.bmm(basis, theta_f32)
            flow_chunks.append(mapped - pts_exp)

        flow = torch.cat(flow_chunks, dim=1)
        flow = flow.view(b, h, w, 2).permute(0, 3, 1, 2).contiguous()
        flow.mul_(valid_f32.view(b, 1, 1, 1))
        return flow

def tps_flow_from_theta(theta, c_src, h, w, device, chunk_size=32768):
    """Hàm tính TPS đơn lẻ (Fallback tương thích ngược cho inference)."""
    n = c_src.shape[0]
    if n < 4 or theta.shape[0] != (n + 3):
        return torch.zeros((1, 2, h, w), device=device)

    pts = get_grid_pts(h, w, device)
    total_pts = pts.shape[0]
    flow_chunks = []

    for i in range(0, total_pts, chunk_size):
        p_chunk = pts[i:i + chunk_size]
        diff = p_chunk.unsqueeze(1) - c_src.unsqueeze(0)
        dist = torch.norm(diff, dim=-1)
        U = tps_rbf(dist)
        P = torch.cat([torch.ones((p_chunk.shape[0], 1), device=device), p_chunk], dim=-1)
        basis = torch.cat([U, P], dim=-1)
        mapped = torch.matmul(basis, theta)
        flow_chunk = mapped - p_chunk
        flow_chunks.append(flow_chunk)

    flow = torch.cat(flow_chunks, dim=0)
    flow = flow.view(h, w, 2).permute(2, 0, 1).unsqueeze(0)
    return flow

print("[✓] CELL 3 hoàn tất: Warp Layers Cache, 8 Anchors & Vectorized TPS Batch Solver sẵn sàng.")

In [ ]:
# [CELL 4 - TỐI ƯU SIÊU TỐC V2.5]: Dataset & DataLoader (Zero Wait Time & Zero RAM Leak)
import os
import random
import cv2
import numpy as np
import pandas as pd
import torch
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torch.utils.data.dataloader import default_collate

class TemporalReversalAugmentation:
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, img0, imggt, img1, t_val, kps=None,
                 c0_pad=None, th01_pad=None, c1_pad=None, th10_pad=None):
        if random.random() < self.p:
            img0_out, img1_out = img1, img0
            t_out = float(1.0 - t_val)
            gt_out = imggt
            kps_out = np.stack([kps[1], kps[0]], axis=0) if (kps is not None and hasattr(kps, "shape") and len(kps.shape) == 3 and kps.shape[0] == 2) else kps
            c0_out, th01_out = c1_pad, th10_pad
            c1_out, th10_out = c0_pad, th01_pad
            return img0_out, gt_out, img1_out, t_out, kps_out, c0_out, th01_out, c1_out, th10_out, True
        return img0, imggt, img1, float(t_val), kps, c0_pad, th01_pad, c1_pad, th10_pad, False

def compute_boundary_motion_weights(df, boost_pct=0.25):
    weights = np.ones(len(df), dtype=np.float32)
    t_vals = df["t"].values.astype(np.float32)
    disp_vals = df["mean_disp"].values.astype(np.float32) if "mean_disp" in df.columns else np.zeros(len(df), dtype=np.float32)
    is_boundary_t = ((t_vals >= 0.15) & (t_vals <= 0.30)) | ((t_vals >= 0.70) & (t_vals <= 0.85))
    is_large_disp = (disp_vals >= 7.0)
    weights[is_boundary_t] *= (1.0 + boost_pct)
    weights[is_large_disp] *= (1.0 + boost_pct)
    return weights

def _worker_init_fn(worker_id):
    configure_glibc_allocator()
    torch.set_num_threads(1)
    cv2.setNumThreads(0)
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

class SakugaGoldenDataset_v1(Dataset):
    def __init__(self, root_dir, df_split, is_train=True, crop_size=512):
        self.root_dir = root_dir
        self.is_train = is_train
        self.crop_size = crop_size
        self.split_name = 'train' if is_train else 'val'
        self.length = len(df_split)
        self.temporal_aug = TemporalReversalAugmentation(p=0.5) if is_train else None

        # 1. Dò tìm đường dẫn gốc 1 lần duy nhất (<0.01s)
        sample_f0 = str(df_split['frame0'].iloc[0]).lstrip('/\\')
        candidate_roots = [
            root_dir,
            os.path.join(root_dir, 'anime_tps_finetune_dataset_ready'),
            os.path.join(root_dir, 'sakuga_golden_dataset_tps_ready_v1_patch2'),
            os.path.join(root_dir, 'sakuga_golden_dataset_tps_ready_v1_patch2', 'anime_tps_finetune_dataset_ready')
        ]
        actual_root = root_dir
        for cr in candidate_roots:
            if not os.path.exists(cr):
                continue
            if os.path.exists(os.path.join(cr, sample_f0)):
                actual_root = cr
                break
            if os.path.exists(os.path.join(cr, self.split_name, sample_f0)):
                actual_root = os.path.join(cr, self.split_name)
                break

        sample_fm = str(df_split['match_file'].iloc[0]).lstrip('/\\')
        actual_match_root = actual_root
        for cr in [actual_root, root_dir] + candidate_roots:
            if not os.path.exists(cr):
                continue
            if os.path.exists(os.path.join(cr, sample_fm)):
                actual_match_root = cr
                break
            if os.path.exists(os.path.join(cr, 'matches', sample_fm)):
                actual_match_root = os.path.join(cr, 'matches')
                break

        f0_list = df_split['frame0'].astype(str).str.lstrip('/\\').tolist()
        fgt_list = df_split['frame_gt'].astype(str).str.lstrip('/\\').tolist()
        f1_list = df_split['frame1'].astype(str).str.lstrip('/\\').tolist()
        fm_list = df_split['match_file'].astype(str).str.lstrip('/\\').tolist()

        p0_bytes = [os.path.join(actual_root, f).encode('utf-8') for f in f0_list]
        pgt_bytes = [os.path.join(actual_root, f).encode('utf-8') for f in fgt_list]
        p1_bytes = [os.path.join(actual_root, f).encode('utf-8') for f in f1_list]
        pmatch_bytes = [os.path.join(actual_match_root, f).encode('utf-8') for f in fm_list]

        self.p0_arr = np.array(p0_bytes, dtype='S256')
        self.pgt_arr = np.array(pgt_bytes, dtype='S256')
        self.p1_arr = np.array(p1_bytes, dtype='S256')
        self.pmatch_arr = np.array(pmatch_bytes, dtype='S256')
        self.t_arr = df_split['t'].to_numpy(dtype=np.float32, copy=True)
        self.clips_arr = np.array([str(c).encode('utf-8') for c in df_split['clip_id']], dtype='S64')

        # 2. Preloading Keypoints đa luồng (~3s)
        MAX_KPS_CACHE = 256
        self.kps_data = np.zeros((self.length, 2, MAX_KPS_CACHE, 2), dtype=np.float32)
        self.kps_count = np.zeros(self.length, dtype=np.int32)

        print(f"[*] Preloading keypoints đa luồng ({self.split_name}: {self.length} mẫu)...", flush=True)

        def _load_single_kps(idx):
            pm = self.pmatch_arr[idx].decode('utf-8')
            try:
                kps = np.load(pm).astype(np.float32)
                if kps.ndim == 3 and kps.shape[0] == 2 and kps.shape[2] == 2:
                    n = kps.shape[1]
                    if n > MAX_KPS_CACHE:
                        idx_sub = np.linspace(0, n - 1, MAX_KPS_CACHE, dtype=int)
                        self.kps_data[idx] = kps[:, idx_sub, :]
                        self.kps_count[idx] = MAX_KPS_CACHE
                    elif n > 0:
                        self.kps_data[idx, :, :n, :] = kps
                        self.kps_count[idx] = n
            except Exception:
                pass

        with ThreadPoolExecutor(max_workers=8) as ex:
            list(tqdm(ex.map(_load_single_kps, range(self.length)), total=self.length, desc=f"   Loading KPS ({self.split_name})", leave=False))
        print(f"[✓] Đã nạp xong keypoints cache ({self.split_name}) vào RAM liền mạch.", flush=True)

        # 3. Precomputing TPS Validation đa luồng (~3s)
        self.val_c0 = np.zeros((self.length, 88, 2), dtype=np.float32)
        self.val_th01 = np.zeros((self.length, 91, 2), dtype=np.float32)
        self.val_c1 = np.zeros((self.length, 88, 2), dtype=np.float32)
        self.val_th10 = np.zeros((self.length, 91, 2), dtype=np.float32)
        self.val_valid = np.zeros((self.length, 1, 1, 1), dtype=np.float32)

        if not self.is_train:
            print(f"[*] Precomputing TPS cache đa luồng cho Validation ({self.length} mẫu)...", flush=True)

            def _solve_single_val_tps(idx):
                p0 = self.p0_arr[idx].decode('utf-8')
                cnt = int(self.kps_count[idx])
                kps = self.kps_data[idx, :, :cnt, :].copy() if cnt > 0 else np.zeros((2, 0, 2), dtype=np.float32)
                try:
                    with Image.open(p0) as img_hdr:
                        w, h = img_hdr.size
                except Exception:
                    w, h = 512, 512
                pad_h = ((h + 31) // 32) * 32
                pad_w = ((w + 31) // 32) * 32
                if kps is not None and kps.shape[1] > 0:
                    kps_norm = kps.copy()
                    kps_norm[0, :, 0] = (kps_norm[0, :, 0] * w) / pad_w
                    kps_norm[0, :, 1] = (kps_norm[0, :, 1] * h) / pad_h
                    kps_norm[1, :, 0] = (kps_norm[1, :, 0] * w) / pad_w
                    kps_norm[1, :, 1] = (kps_norm[1, :, 1] * h) / pad_h
                else:
                    kps_norm = kps
                c0_p, th01_p, c1_p, th10_p, v = solve_tps_pair_cpu(kps_norm, max_kps=80, lambd=0.5, min_dist=0.015, return_tensor=False)
                self.val_c0[idx] = c0_p
                self.val_th01[idx] = th01_p
                self.val_c1[idx] = c1_p
                self.val_th10[idx] = th10_p
                self.val_valid[idx, 0, 0, 0] = float(v)

            with ThreadPoolExecutor(max_workers=8) as ex:
                list(tqdm(ex.map(_solve_single_val_tps, range(self.length)), total=self.length, desc="   Solving Val TPS", leave=False))
            print(f"[✓] Đã precompute xong {self.length} mẫu TPS Validation vào contiguous NumPy buffers.", flush=True)

        print(f"[✓] Dataset ({self.split_name}: {self.length} mẫu) sẵn sàng (Khởi động tức thì < 5s, Zero RAM Leak).", flush=True)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        p0 = self.p0_arr[idx].decode('utf-8')
        pgt = self.pgt_arr[idx].decode('utf-8')
        p1 = self.p1_arr[idx].decode('utf-8')
        clip_name = int(idx) if self.is_train else self.clips_arr[idx].decode('utf-8')

        im0 = cv2.imread(p0, cv2.IMREAD_GRAYSCALE)
        imgt = cv2.imread(pgt, cv2.IMREAD_GRAYSCALE)
        im1 = cv2.imread(p1, cv2.IMREAD_GRAYSCALE)

        if im0 is None or imgt is None or im1 is None:
            cs = self.crop_size if self.is_train else 512
            im0 = np.full((cs, cs), 255, dtype=np.uint8) if im0 is None else im0
            imgt = np.full((cs, cs), 255, dtype=np.uint8) if imgt is None else imgt
            im1 = np.full((cs, cs), 255, dtype=np.uint8) if im1 is None else im1

        h, w = im0.shape

        if self.is_train:
            cnt = int(self.kps_count[idx])
            kps = self.kps_data[idx, :, :cnt, :].copy() if cnt > 0 else np.zeros((2, 0, 2), dtype=np.float32)
            cs = self.crop_size
            if h > cs and w > cs:
                top = random.randint(0, h - cs)
                left = random.randint(0, w - cs)
                im0_crop = im0[top:top+cs, left:left+cs]
                imggt_crop = imgt[top:top+cs, left:left+cs]
                im1_crop = im1[top:top+cs, left:left+cs]

                if kps.shape[1] > 0:
                    kps[0, :, 0] = (kps[0, :, 0] * w - left) / cs
                    kps[0, :, 1] = (kps[0, :, 1] * h - top) / cs
                    kps[1, :, 0] = (kps[1, :, 0] * w - left) / cs
                    kps[1, :, 1] = (kps[1, :, 1] * h - top) / cs
                    mask0 = (kps[0, :, 0] >= 0) & (kps[0, :, 0] <= 1) & (kps[0, :, 1] >= 0) & (kps[0, :, 1] <= 1)
                    mask1 = (kps[1, :, 0] >= 0) & (kps[1, :, 0] <= 1) & (kps[1, :, 1] >= 0) & (kps[1, :, 1] <= 1)
                    kps = kps[:, mask0 & mask1, :]
            else:
                pad_h = ((h + 31) // 32) * 32
                pad_w = ((w + 31) // 32) * 32
                im0_crop = np.pad(im0, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)
                imggt_crop = np.pad(imgt, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)
                im1_crop = np.pad(im1, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)

            img0 = (255 - im0_crop).astype(np.float32) * (1.0 / 255.0)
            imggt = (255 - imggt_crop).astype(np.float32) * (1.0 / 255.0)
            img1 = (255 - im1_crop).astype(np.float32) * (1.0 / 255.0)

            c0_pad, th01_pad, c1_pad, th10_pad, valid_flag = solve_tps_pair_cpu(kps, max_kps=80, lambd=0.5, min_dist=0.015, return_tensor=False)
            valid_flag = np.full((1, 1, 1), float(valid_flag), dtype=np.float32)

            t_scalar = float(self.t_arr[idx])
            if self.temporal_aug is not None:
                img0, imggt, img1, t_scalar, kps, c0_pad, th01_pad, c1_pad, th10_pad, _ = self.temporal_aug(
                    img0, imggt, img1, t_scalar, kps, c0_pad, th01_pad, c1_pad, th10_pad
                )
        else:
            pad_h = ((h + 31) // 32) * 32
            pad_w = ((w + 31) // 32) * 32
            im0_pad = np.pad(im0, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)
            imggt_pad = np.pad(imgt, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)
            im1_pad = np.pad(im1, ((0, pad_h - h), (0, pad_w - w)), mode='constant', constant_values=255)

            img0 = (255 - im0_pad).astype(np.float32) * (1.0 / 255.0)
            imggt = (255 - imggt_pad).astype(np.float32) * (1.0 / 255.0)
            img1 = (255 - im1_pad).astype(np.float32) * (1.0 / 255.0)

            c0_pad = self.val_c0[idx].copy()
            th01_pad = self.val_th01[idx].copy()
            c1_pad = self.val_c1[idx].copy()
            th10_pad = self.val_th10[idx].copy()
            valid_flag = self.val_valid[idx].copy()
            t_scalar = float(self.t_arr[idx])

        t0 = np.ascontiguousarray(img0[None, :, :], dtype=np.float32)
        tgt = np.ascontiguousarray(imggt[None, :, :], dtype=np.float32)
        t1 = np.ascontiguousarray(img1[None, :, :], dtype=np.float32)
        t_val = np.full((1, 1, 1), t_scalar, dtype=np.float32)
        c0_pad = np.ascontiguousarray(c0_pad, dtype=np.float32)
        th01_pad = np.ascontiguousarray(th01_pad, dtype=np.float32)
        c1_pad = np.ascontiguousarray(c1_pad, dtype=np.float32)
        th10_pad = np.ascontiguousarray(th10_pad, dtype=np.float32)
        valid_flag = np.ascontiguousarray(valid_flag, dtype=np.float32)

        return t0, tgt, t1, t_val, c0_pad, th01_pad, c1_pad, th10_pad, valid_flag, clip_name

custom_collate_fn_v1 = default_collate

PER_GPU_BATCH = 8
BATCH_SIZE = PER_GPU_BATCH * max(1, NUM_GPUS)
VAL_BATCH_SIZE = max(4, NUM_GPUS * 2)

if 'train_df' not in globals() or train_df is None:
    print("[*] Tự động nạp lại train_df/val_df cho Cell 4...")
    _meta_path = os.path.join(DATASET_ROOT, 'triplets.csv')
    if not os.path.exists(_meta_path):
        _meta_path = os.path.join(DATASET_ROOT, 'triplets_combined.csv')
    _df = pd.read_csv(_meta_path)
    train_df = _df[_df['split'] == 'train'].reset_index(drop=True)
    val_df = _df[_df['split'] == 'val'].reset_index(drop=True)
    del _df

train_dataset = SakugaGoldenDataset_v1(DATASET_ROOT, train_df, is_train=True, crop_size=512)
val_dataset = SakugaGoldenDataset_v1(DATASET_ROOT, val_df, is_train=False)

train_sample_weights = compute_boundary_motion_weights(train_df, boost_pct=0.25)
train_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(train_sample_weights, dtype=torch.double),
    num_samples=len(train_dataset),
    replacement=True
)

del train_df, val_df
cleanup_memory()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True,
    prefetch_factor=2,
    collate_fn=custom_collate_fn_v1,
    worker_init_fn=_worker_init_fn,
    drop_last=True
)

val_loader_full = DataLoader(
    val_dataset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    persistent_workers=True,
    pin_memory=True,
    prefetch_factor=2,
    collate_fn=custom_collate_fn_v1,
    worker_init_fn=_worker_init_fn
)

val_loader = val_loader_full

print(f"[✓] DataLoaders Golden Dataset V2.5 sẵn sàng: {len(train_loader)} batches (Train), {len(val_loader_full)} batches (Val Full - Batch {VAL_BATCH_SIZE}).")
print("[✓] CELL 4 hoàn tất: Khởi động tức thì (< 5s), Zero RAM Leak & Tận dụng 4 vCPU Streaming.")
print(f"[*] Host RAM hiện tại -> {format_host_ram()}")

In [ ]:
# [CELL 5]: Kiến Trúc TPS-Inbetween V2.5 (ASF-GR Engine - Không Nghẽn Cổ Chai GPU)
def conv(in_p, out_p, kernel_size=3, stride=1, padding=1, dilation=1):
    return nn.Sequential(
        nn.Conv2d(in_p, out_p, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, bias=True),
        nn.PReLU(out_p)
    )

def deconv(in_p, out_p, kernel_size=4, stride=2, padding=1):
    return nn.Sequential(
        nn.ConvTranspose2d(in_p, out_p, kernel_size=kernel_size, stride=stride, padding=padding, bias=True),
        nn.PReLU(out_p)
    )

class Conv2(nn.Module):
    def __init__(self, in_p, out_p, stride=2):
        super(Conv2, self).__init__()
        self.conv1 = conv(in_p, out_p, 3, stride, 1)
        self.conv2 = conv(out_p, out_p, 3, 1, 1)
    def forward(self, x):
        return self.conv2(self.conv1(x))

class Contextnet(nn.Module):
    def __init__(self, indim=1, c=16):
        super(Contextnet, self).__init__()
        self.conv1 = Conv2(indim, c)
        self.conv2 = Conv2(c, 2*c)
        self.conv3 = Conv2(2*c, 4*c)
        self.conv4 = Conv2(4*c, 8*c)
    def forward(self, x, flow):
        x = self.conv1(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f1 = warp_normalized_flow(x, flow)
        x = self.conv2(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f2 = warp_normalized_flow(x, flow)
        x = self.conv3(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f3 = warp_normalized_flow(x, flow)
        x = self.conv4(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f4 = warp_normalized_flow(x, flow)
        return [f1, f2, f3, f4]

class IFBlock(nn.Module):
    def __init__(self, in_planes, c=64):
        super(IFBlock, self).__init__()
        self.conv0 = nn.Sequential(
            conv(in_planes, c//2, 3, 2, 1),
            conv(c//2, c, 3, 2, 1),
        )
        self.convblock = nn.Sequential(*[conv(c, c) for _ in range(8)])
        self.lastconv = nn.ConvTranspose2d(c, 5, 4, 2, 1)

    def forward(self, x, flow, scale):
        if scale != 1:
            x = F.interpolate(x, scale_factor=1./scale, mode='bilinear', align_corners=False)
        if flow is not None:
            flow = F.interpolate(flow, scale_factor=1./scale, mode='bilinear', align_corners=False) * (1./scale)
            x = torch.cat((x, flow), 1)
        x = self.conv0(x)
        x = self.convblock(x) + x
        tmp = self.lastconv(x)
        tmp = F.interpolate(tmp, scale_factor=scale * 2, mode='bilinear', align_corners=False)
        flow = tmp[:, :4] * scale * 2
        mask = tmp[:, 4:5]
        return flow, mask

class IFNet(nn.Module):
    def __init__(self):
        super(IFNet, self).__init__()
        self.block0 = IFBlock(6, c=240)
        self.block1 = IFBlock(13+4, c=150)
        self.block2 = IFBlock(13+4, c=90)

    def forward(self, I0t, I1t, scale=[4, 2, 1]):
        x = torch.cat((I0t.repeat(1, 3, 1, 1), I1t.repeat(1, 3, 1, 1)), dim=1)
        img0 = x[:, :3]
        img1 = x[:, 3:6]
        flow_list = []
        mask_list = []
        warped_img0 = img0
        warped_img1 = img1
        flow = None
        stu = [self.block0, self.block1, self.block2]
        for i in range(3):
            if flow is not None:
                flow_d, mask_d = stu[i](torch.cat((img0, img1, warped_img0, warped_img1, mask), 1), flow, scale=scale[i])
                flow = flow + flow_d
                mask = mask + mask_d
            else:
                flow, mask = stu[i](torch.cat((img0, img1), 1), None, scale=scale[i])
            mask_list.append(torch.sigmoid(mask))
            flow_list.append(flow)
            warped_img0 = warp_pixel_flow(img0, flow[:, :2])
            warped_img1 = warp_pixel_flow(img1, flow[:, 2:4])
        return flow_list[2], mask_list[2]

# [ĐỘT PHÁ 1]: Adaptive Stroke Fusion Head (ASF Head) - Triệt tiêu Hard Blending
class AdaptiveStrokeFusionHead(nn.Module):
    def __init__(self):
        super(AdaptiveStrokeFusionHead, self).__init__()
        self.net = nn.Sequential(
            conv(8, 32, 3, 1, 1),
            conv(32, 32, 3, 1, 1),
            nn.Conv2d(32, 1, 3, 1, 1)
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, fine_I0t, fine_I1t, flow, ifnet_mask, t_vals):
        b, _, h, w = fine_I0t.shape
        t_map = t_vals.repeat(1, 1, h, w)
        inp = torch.cat([fine_I0t, fine_I1t, flow, ifnet_mask, t_map], dim=1)
        residual_logit = self.net(inp)
        prior_logit = torch.logit(torch.clamp(1.0 - t_map, 1e-4, 1.0 - 1e-4))
        blend_mask = torch.sigmoid(prior_logit + residual_logit)
        return blend_mask

# [ĐỘT PHÁ 2]: Synthesis UNet v1.1 với Gated Ghost Erasure (GGER) & Unlocked Detail Add
class SynthesisUnet_v1(nn.Module):
    def __init__(self, indim=9, c=16):
        super(SynthesisUnet_v1, self).__init__()
        self.down0 = Conv2(indim, 2*c)
        self.down1 = Conv2(4*c, 4*c)
        self.down2 = Conv2(8*c, 8*c)
        self.down3 = Conv2(16*c, 16*c)
        self.up0 = deconv(32*c, 8*c)
        self.up1 = deconv(16*c, 4*c)
        self.up2 = deconv(8*c, 2*c)
        self.up3 = deconv(4*c, c)

        self.conv_erase = nn.Conv2d(c, 1, 3, 1, 1)
        nn.init.zeros_(self.conv_erase.weight)
        nn.init.constant_(self.conv_erase.bias, 2.0)  # sigmoid(2.0) ~ 0.88

        self.conv_add = nn.Conv2d(c, 1, 3, 1, 1)
        nn.init.zeros_(self.conv_add.weight)
        nn.init.constant_(self.conv_add.bias, -3.0)

    def forward(self, img0, img1, flow0, flow1, warped0, warped1, mask, c0, c1):
        s0 = self.down0(torch.cat((img0, img1, warped0, warped1, flow0, flow1, mask), 1))
        s1 = self.down1(torch.cat((s0, c0[0], c1[0]), 1))
        s2 = self.down2(torch.cat((s1, c0[1], c1[1]), 1))
        s3 = self.down3(torch.cat((s2, c0[2], c1[2]), 1))
        x = self.up0(torch.cat((s3, c0[3], c1[3]), 1))
        x = self.up1(torch.cat((x, s2), 1))
        x = self.up2(torch.cat((x, s1), 1))
        x = self.up3(torch.cat((x, s0), 1))

        gate_erase = torch.sigmoid(self.conv_erase(x))
        detail_add = torch.sigmoid(self.conv_add(x))
        return gate_erase, detail_add

class TPSInbetweenFinetune_v1(nn.Module):
    """
    Kiến trúc TPS-Inbetween V2.5 hoàn chỉnh.
    Bảo toàn 100% cấu trúc state_dict (encoder, IFNet, asf_head, unet) để tương thích tuyệt đối.
    """
    def __init__(self):
        super(TPSInbetweenFinetune_v1, self).__init__()
        self.encoder = Contextnet(indim=1)
        self.IFNet = IFNet()
        self.asf_head = AdaptiveStrokeFusionHead()
        self.unet = SynthesisUnet_v1(indim=9)

    def forward(self, x0, x1, t_vals, kps_input=None, kps_counts=None, max_kps=80, tps_precomputed=None):
        b, c, h, w = x0.shape
        dev = x0.device
        t_vals = t_vals.view(b, 1, 1, 1).to(dev)

        # 1. Coarse TPS Motion
        # [TỐI ƯU HÓA ĐỘT PHÁ V2.5]: Nếu đã có tps_precomputed từ DataLoader -> Chạy 100% GPU BMM Batch
        if tps_precomputed is not None:
            c0_b, th01_b, c1_b, th10_b, valid_b = tps_precomputed
            c0_b = c0_b.to(dev)
            th01_b = th01_b.to(dev)
            c1_b = c1_b.to(dev)
            th10_b = th10_b.to(dev)
            valid_b = valid_b.to(dev)

            coarse_flow01 = tps_flow_from_theta_batch(th01_b, c0_b, valid_b, h, w, dev)
            coarse_flow10 = tps_flow_from_theta_batch(th10_b, c1_b, valid_b, h, w, dev)
        else:
            # Nhánh Fallback tương thích ngược cho inference bên ngoài
            flow01_list, flow10_list = [], []
            with torch.no_grad():
                for i in range(b):
                    kps = kps_input[i] if kps_input is not None else None
                    c0_p, th01_p, c1_p, th10_p, v = solve_tps_pair_cpu(kps, max_kps=max_kps, lambd=0.5)
                    c0_p = c0_p.unsqueeze(0).to(dev)
                    th01_p = th01_p.unsqueeze(0).to(dev)
                    c1_p = c1_p.unsqueeze(0).to(dev)
                    th10_p = th10_p.unsqueeze(0).to(dev)
                    v_t = v.view(1, 1, 1, 1).to(dev)

                    f01 = tps_flow_from_theta_batch(th01_p, c0_p, v_t, h, w, dev)
                    f10 = tps_flow_from_theta_batch(th10_p, c1_p, v_t, h, w, dev)
                    flow01_list.append(f01)
                    flow10_list.append(f10)

            coarse_flow01 = torch.cat(flow01_list, dim=0)
            coarse_flow10 = torch.cat(flow10_list, dim=0)

        coarse_flow0t = coarse_flow01 * (t_vals ** 2) - (1.0 - t_vals) * t_vals * coarse_flow10
        coarse_flow1t = coarse_flow10 * ((1.0 - t_vals) ** 2) - t_vals * (1.0 - t_vals) * coarse_flow01

        # 2. Coarse Warping
        coarse_I0t = warp_normalized_flow(x0, coarse_flow0t)
        coarse_I1t = warp_normalized_flow(x1, coarse_flow1t)

        # 3. Fine Motion Refinement (Cho phép gradient truyền qua Contextnet khi fine-tune V2.5)
        feats0 = self.encoder(x0, coarse_flow0t)
        feats1 = self.encoder(x1, coarse_flow1t)
        flow_refine, ifnet_mask = self.IFNet(coarse_I0t, coarse_I1t)

        # 4. Fine Warping
        fine_I0t = warp_pixel_flow(coarse_I0t, flow_refine[:, :2])
        fine_I1t = warp_pixel_flow(coarse_I1t, flow_refine[:, 2:4])

        # 5. Adaptive Stroke Fusion (ASF Head)
        blend_mask = self.asf_head(fine_I0t, fine_I1t, flow_refine, ifnet_mask, t_vals)
        base_interp = blend_mask * fine_I0t + (1.0 - blend_mask) * fine_I1t

        # 6. Gated Ghost Erasure Residual (Synthesis UNet v1.0)
        gate_erase, detail_add = self.unet(
            x0, x1,
            coarse_flow0t, coarse_flow1t,
            fine_I0t, fine_I1t, ifnet_mask, feats0, feats1
        )
        pred_t = torch.clamp(gate_erase * base_interp + detail_add, 0.0, 1.0)
        return pred_t

base_model = TPSInbetweenFinetune_v1().to(device)
print("[*] Mô hình TPSInbetweenFinetune_v1 (ASF-GR Engine V2.5 - Unlocked Detail Add) đã khởi tạo thành công.")

In [ ]:
# [CELL 6]: Bộ Hàm Mất Mát CompositeLossEngine_v25 (Exact-STE Density Calibration & Bidirectional Stroke Width Control)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class StrokeDensityRegularizationLoss(nn.Module):
    """
    Hàm mất mát điều tiết mật độ mực nét vẽ V2.5 (Exact-STE & Non-Saturating Leaky Ramp):
    - Đánh giá bit-exact tại đúng ngưỡng binary threshold = 0.15 thông qua Straight-Through Estimator (STE).
    - Khử 100% nhiễu nền trắng (p_ink <= 0.06 cho đúng 0.0 tuyệt đối).
    - Đạo hàm dL/dp không bao giờ bằng 0 khi phình nét nhờ nhánh Leaky Surrogate Ramp (+ 0.25 * relu(p - 0.24)).
    - Dead-zone chuẩn dải [target_min=0.96, target_max=1.03] (tâm chuẩn lý tưởng 1.00).
    - Phạt nghiêm ngặt khi phình nét (dev_high factor: 2.5x + 1.5 * dev_high^2).
    """
    def __init__(self, target_min=0.96, target_max=1.03, eps=1.0):
        super(StrokeDensityRegularizationLoss, self).__init__()
        self.target_min = target_min
        self.target_max = target_max
        self.eps = eps

    def forward(self, pred, target, ink_mode=True):
        p_ink = (pred if ink_mode else (1.0 - pred)).float()
        g_ink = (target if ink_mode else (1.0 - target)).float()

        # 1. Đo lường diện tích nét nhị phân chuẩn tại ngưỡng 0.15 (bit-exact)
        m_hard_p = (p_ink > 0.15).float()
        m_hard_g = (g_ink > 0.15).float()

        # 2. Xấp xỉ khả vi (Surrogate Soft Mask) không bị bão hòa đạo hàm ở vùng nét dày:
        # Nền trắng (p_ink <= 0.06) -> hoàn toàn 0.0
        # Vùng chuyển tiếp [0.06, 0.24] -> dốc chuẩn (p - 0.06) / 0.18
        # Vùng mực dày (p_ink > 0.24) -> cộng thêm nhánh leaky + 0.25 * relu(p - 0.24) để đạo hàm luôn dương dL/dp > 0
        m_soft_p = torch.clamp((p_ink - 0.06) / 0.18, 0.0, 1.0) + 0.25 * F.relu(p_ink - 0.24)
        m_soft_g = torch.clamp((g_ink - 0.06) / 0.18, 0.0, 1.0) + 0.25 * F.relu(g_ink - 0.24)

        # 3. Straight-Through Estimator (STE): Forward lấy m_hard_p, Backward truyền qua m_soft_p
        m_ste_p = m_hard_p.detach() + (m_soft_p - m_soft_p.detach())
        m_ste_g = m_hard_g.detach() + (m_soft_g - m_soft_g.detach())

        sum_p_mask = torch.sum(m_ste_p, dim=[-3, -2, -1])
        sum_g_mask = torch.sum(m_ste_g, dim=[-3, -2, -1])

        valid_gt = torch.sum(m_hard_g, dim=[-3, -2, -1]) > 20.0
        ratio_mask = torch.where(valid_gt, sum_p_mask / (sum_g_mask + self.eps), torch.ones_like(sum_p_mask))
        ratio_mask = torch.clamp(ratio_mask, 0.0, 10.0)

        # 4. Tỷ lệ tổng lượng mực liên tục (khử nền p <= 0.05)
        sum_p_raw = torch.sum(F.relu(p_ink - 0.05), dim=[-3, -2, -1])
        sum_g_raw = torch.sum(g_ink, dim=[-3, -2, -1])
        ratio_raw = torch.where(valid_gt, sum_p_raw / (sum_g_raw + self.eps), torch.ones_like(sum_p_raw))
        ratio_raw = torch.clamp(ratio_raw, 0.0, 10.0)

        ratio = 0.80 * ratio_mask + 0.20 * ratio_raw

        # 5. Phạt đối xứng với trọng tâm siết phình nét (dev_high)
        dev_low = F.relu(self.target_min - ratio)
        dev_high = F.relu(ratio - self.target_max)
        loss_vec = 1.5 * dev_low + 2.5 * dev_high + 1.5 * (dev_high ** 2)
        loss = loss_vec.mean()

        # Báo cáo tỷ lệ diện tích nét thực tế đo được
        sum_p_actual = torch.sum(m_hard_p, dim=[-3, -2, -1])
        sum_g_actual = torch.sum(m_hard_g, dim=[-3, -2, -1])
        actual_ratio = torch.where(valid_gt, sum_p_actual / (sum_g_actual + self.eps), torch.ones_like(sum_p_actual))

        return loss, actual_ratio.mean().detach()

class DirectionalTangentCoherenceLoss(nn.Module):
    def __init__(self, eps=1e-4):
        super(DirectionalTangentCoherenceLoss, self).__init__()
        self.eps = eps
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3) / 8.0
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3) / 8.0
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    def forward(self, pred, target):
        dev = pred.device
        gx_p = F.conv2d(pred, self.sobel_x.to(dtype=pred.dtype, device=dev), padding=1).float()
        gy_p = F.conv2d(pred, self.sobel_y.to(dtype=pred.dtype, device=dev), padding=1).float()
        gx_g = F.conv2d(target, self.sobel_x.to(dtype=target.dtype, device=dev), padding=1).float()
        gy_g = F.conv2d(target, self.sobel_y.to(dtype=target.dtype, device=dev), padding=1).float()

        with torch.amp.autocast('cuda', enabled=False):
            mag_p = torch.sqrt(gx_p * gx_p + gy_p * gy_p + self.eps)
            mag_g = torch.sqrt(gx_g * gx_g + gy_g * gy_g + self.eps)
            cos_sim = (gx_p * gx_g + gy_p * gy_g) / (mag_p * mag_g)
            loss_pixel = 1.0 - torch.abs(cos_sim)
            mask_f = ((mag_g > 0.02) | (target.float() > 0.10)).float()
            return torch.sum(loss_pixel * mask_f) / (torch.sum(mask_f) + 1.0)

class SoftClDiceLoss(nn.Module):
    def __init__(self, iters=5, eps=1e-5, beta_sensitivity=0.50):
        super(SoftClDiceLoss, self).__init__()
        self.iters = iters
        self.eps = eps
        self.beta = beta_sensitivity

    def soft_erode(self, img):
        return -F.max_pool2d(-img, kernel_size=3, stride=1, padding=1)

    def soft_open(self, img):
        return F.max_pool2d(self.soft_erode(img), kernel_size=3, stride=1, padding=1)

    def soft_skeletonize(self, img):
        skel = torch.zeros_like(img)
        x = img
        opened_first = None
        for i in range(self.iters):
            opened = self.soft_open(x)
            if i == 0:
                opened_first = opened
            sub = F.relu(x - opened)
            skel = torch.max(skel, sub)
            x = self.soft_erode(x)
        return skel, opened_first

    def forward(self, pred, target):
        skel_pred, opened_pred = self.soft_skeletonize(pred)
        skel_gt, opened_gt = self.soft_skeletonize(target)
        tprec = (torch.sum(skel_pred * target, dim=[-3, -2, -1]) + self.eps) / (torch.sum(skel_pred, dim=[-3, -2, -1]) + self.eps)
        tsens = (torch.sum(skel_gt * pred, dim=[-3, -2, -1]) + self.eps) / (torch.sum(skel_gt, dim=[-3, -2, -1]) + self.eps)
        # SoftClDice cân bằng 50/50 giữa Precision và Sensitivity (beta=0.50)
        cldice = (tprec * tsens) / (self.beta * tprec + (1.0 - self.beta) * tsens + self.eps)
        return torch.mean(1.0 - cldice), skel_gt, opened_gt

class BidirectionalStrokeWidthLoss(nn.Module):
    """
    Hàm mất mát kiểm soát bề rộng nét hai chiều V2.5:
    1. Nhánh 1 (loss_under): Bảo tồn lõi nét mỏng và khung xương trên vùng nét thật của Ground Truth.
    2. Nhánh 2 (loss_over_edge): Phạt nghiêm ngặt tràn mực (overspill) trên vành đai viền 1-2px quanh nét thật.
    """
    def __init__(self, eps=1e-5):
        super(BidirectionalStrokeWidthLoss, self).__init__()
        self.eps = eps

    def forward(self, pred, target, opened_gt, skel_gt):
        p_ink = pred.float()
        g_ink = target.float()

        gt_core = (g_ink > 0.15).float()

        # Nhánh 1: Bảo tồn lõi nét mỏng (Thin Stroke & Skeleton Preservation)
        thin_mask = torch.clamp(g_ink - opened_gt, 0.0, 1.0)
        core_weight = 1.0 + 2.5 * thin_mask + 1.5 * skel_gt
        under_pen = F.relu(g_ink - p_ink) * core_weight
        loss_under = torch.sum(under_pen * gt_core) / (torch.sum(gt_core) + 1.0)

        # Nhánh 2: Chống phình nét trên vành đai viền 1-2px quanh nét vẽ thật (Edge Ring Overspill Penalty)
        gt_dilated_3x3 = F.max_pool2d(gt_core, kernel_size=3, stride=1, padding=1)
        gt_dilated_5x5 = F.max_pool2d(gt_core, kernel_size=5, stride=1, padding=2)
        edge_ring = torch.clamp(gt_dilated_5x5 - gt_core, 0.0, 1.0)

        # Phạt khi dự đoán mực vượt quá ngưỡng cho phép tại vành đai viền
        over_spill = F.relu(p_ink - torch.maximum(g_ink, torch.tensor(0.08, device=p_ink.device)))
        edge_bloat = F.relu(p_ink - 0.14)
        loss_over_edge = torch.sum((over_spill + 2.0 * edge_bloat) * edge_ring) / (torch.sum(edge_ring) + 1.0)

        return loss_under + 1.25 * loss_over_edge

# Alias tương thích
FineStrokePreservationLoss = BidirectionalStrokeWidthLoss

class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super(VGGPerceptualLoss, self).__init__()
        vgg_features = models.vgg16(pretrained=True).features
        self.slice1 = nn.Sequential(*[vgg_features[x] for x in range(4)])
        self.slice2 = nn.Sequential(*[vgg_features[x] for x in range(4, 9)])
        del vgg_features
        gc.collect()
        for param in self.parameters():
            param.requires_grad = False
        self.eval()

    def forward(self, pred, target):
        pred_3c = pred.repeat(1, 3, 1, 1)
        target_3c = target.repeat(1, 3, 1, 1)
        h_pred1 = self.slice1(pred_3c)
        h_target1 = self.slice1(target_3c)
        loss = F.l1_loss(h_pred1, h_target1)
        h_pred2 = self.slice2(h_pred1)
        h_target2 = self.slice2(h_target1)
        loss += F.l1_loss(h_pred2, h_target2)
        return loss

class StrokeDiceLoss(nn.Module):
    def __init__(self, eps=1e-5):
        super(StrokeDiceLoss, self).__init__()
        self.eps = eps

    def forward(self, pred, target):
        intersection = 2.0 * torch.sum(pred * target, dim=[1, 2, 3]) + self.eps
        union = torch.sum(pred ** 2, dim=[1, 2, 3]) + torch.sum(target ** 2, dim=[1, 2, 3]) + self.eps
        dice = intersection / union
        return torch.mean(1.0 - dice)

class GPUMorphologicalDistanceLoss(nn.Module):
    def __init__(self):
        super(GPUMorphologicalDistanceLoss, self).__init__()
        self.scales = [3, 5, 9, 15]

    def forward(self, pred, target):
        p_bin = (pred > 0.15).float()
        t_bin = (target > 0.15).float()

        loss_accum = 0.0
        for k in self.scales:
            pad = k // 2
            t_dilated = F.max_pool2d(t_bin, kernel_size=k, stride=1, padding=pad)
            p_dilated = F.max_pool2d(p_bin, kernel_size=k, stride=1, padding=pad)

            loss_p2t = torch.mean(pred * (1.0 - t_dilated))
            loss_t2p = torch.mean(target * (1.0 - p_dilated))
            loss_accum += (loss_p2t + loss_t2p)
        return loss_accum / len(self.scales)

class GhostStrokePenaltyLoss(nn.Module):
    def __init__(self, radius=5):
        super(GhostStrokePenaltyLoss, self).__init__()
        self.radius = radius

    def forward(self, pred, target):
        pred_f = pred.float()
        target_f = target.float()
        pad = self.radius // 2
        # Vùng ngoài 5x5: phạt bóng ma xa
        t_dilated = F.max_pool2d((target_f > 0.15).float(), kernel_size=self.radius, stride=1, padding=pad)
        outside_dilated = 1.0 - t_dilated
        far_ghost = F.relu(pred_f - 0.03) * outside_dilated

        # Vùng cận biên nằm ngoài nét thật: phạt quầng xám > 0.10 mà không xâm lấn lõi nét target_f >= 0.15
        outside_core = (target_f < 0.10).float()
        near_halo = F.relu(pred_f - 0.10) * outside_core

        return torch.mean(far_ghost ** 2) + 0.5 * torch.mean(near_halo)

class CompositeLossEngine_v25(nn.Module):
    def __init__(self):
        super(CompositeLossEngine_v25, self).__init__()
        self.vgg_loss_fn = VGGPerceptualLoss()
        self.dice_loss_fn = StrokeDiceLoss()
        self.morph_dist_fn = GPUMorphologicalDistanceLoss()
        self.ghost_loss_fn = GhostStrokePenaltyLoss(radius=5)
        self.density_loss_fn = StrokeDensityRegularizationLoss(target_min=0.96, target_max=1.03, eps=1.0)
        self.tangent_loss_fn = DirectionalTangentCoherenceLoss(eps=1e-4)
        self.cldice_loss_fn = SoftClDiceLoss(iters=5, beta_sensitivity=0.50)
        self.fine_loss_fn = BidirectionalStrokeWidthLoss()

    def forward(self, pred, target):
        loss_l1 = F.l1_loss(pred, target)
        loss_vgg = self.vgg_loss_fn(pred, target)
        loss_dice = self.dice_loss_fn(pred, target)
        loss_morph = self.morph_dist_fn(pred, target)
        loss_ghost = self.ghost_loss_fn(pred, target)
        loss_density, density_ratio = self.density_loss_fn(pred, target, ink_mode=True)
        loss_tangent = self.tangent_loss_fn(pred, target)
        loss_cldice, skel_gt, opened_gt = self.cldice_loss_fn(pred, target)
        loss_fine = self.fine_loss_fn(pred, target, opened_gt, skel_gt)

        total_loss = (
            1.00 * loss_l1 +
            0.15 * loss_vgg +
            0.30 * loss_dice +
            0.12 * loss_morph +
            0.25 * loss_ghost +
            0.55 * loss_density +
            0.20 * loss_tangent +
            0.25 * loss_cldice +
            0.25 * loss_fine
        )
        return total_loss, loss_l1, loss_dice, loss_density, loss_fine

# Tương thích ngược alias
CompositeLossEngine_v24 = CompositeLossEngine_v25
CompositeLossEngine_v23 = CompositeLossEngine_v25
CompositeLossEngine_v22 = CompositeLossEngine_v25

loss_engine = CompositeLossEngine_v25().to(device)
print("[*] Bộ Loss CompositeLossEngine_v25 Cân Bằng Nét Mảnh & Anti-Thickening đã khởi tạo thành công.")

In [ ]:
# [CELL 7]: Optimizer, Multi-GPU Parallel Training Wrapper & Nạp Checkpoint (Dual-Checkpoint Model Soup V2.5)
for param in base_model.parameters():
    param.requires_grad = True

# Danh sách ứng viên Checkpoint Dày nét (Thick Models: Density ~ 1.3 - 1.5)
thick_candidates = [
    os.path.join(KAGGLE_INPUT, 'tps-v24-artifacts', 'model_tps_v2.4_best.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-v24-artifacts', 'model_tps_v2.4_best_progress.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-v24-checkpoint', 'model_tps_v2.4_best.pt'),
    os.path.join(WORKING_DIR, 'checkpoints_v2.4', 'model_tps_v2.4_best.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-v20-artifacts', 'model_tps_v2.0_best.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-v20-checkpoint', 'model_tps_v2.0_best.pt'),
    os.path.join(WORKING_DIR, 'checkpoints_v2.0.2', 'model_tps_v2.0_best.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-v22-artifacts', 'model_tps_v2.2_best.pt'),
    os.path.join(WORKING_DIR, 'checkpoints_v2.2', 'model_tps_v2.2_best.pt'),
]

# Danh sách ứng viên Checkpoint Mỏng nét (Thin Models: Density ~ 0.68)
thin_candidates = [
    os.path.join(KAGGLE_INPUT, 'tps-v23-artifacts', 'model_tps_v2.3_best.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-v23-artifacts', 'model_tps_v2.3_best_progress.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-v23-checkpoint', 'model_tps_v2.3_best.pt'),
    os.path.join(WORKING_DIR, 'checkpoints_v2.3', 'model_tps_v2.3_best.pt'),
    os.path.join(WORKING_DIR, 'checkpoints_v2.3', 'model_tps_v2.3_interrupted.pt'),
]

# Danh sách ứng viên bổ trợ nếu không tìm thấy theo nhóm
fallback_candidates = [
    os.path.join(WORKING_DIR, 'checkpoints_v1.1', 'model_tps_v1.1_best.pt'),
    os.path.join(WORKING_DIR, 'checkpoints_v1.0', 'model_tps_v1.0_best.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-baseline-model', 'model_latest.pt')
]

thick_found = None
thin_found = None

for cp in thick_candidates:
    if os.path.exists(cp):
        thick_found = cp
        break

for cp in thin_candidates:
    if os.path.exists(cp):
        thin_found = cp
        break

def _extract_sd(cp_path):
    sd = torch.load(cp_path, map_location='cpu')
    if isinstance(sd, dict):
        if 'state_dict' in sd:
            sd = sd['state_dict']
        elif 'model' in sd:
            sd = sd['model']
    return sd

loaded_ckpt = False

# Cơ chế 1: Dual-Checkpoint Model Soup Blending (52% Thick + 48% Thin)
if thick_found and thin_found:
    try:
        print(f"[*] Tìm thấy CẢ HAI checkpoint: Thick ({thick_found}) & Thin ({thin_found})")
        print("[*] Đang thực hiện pha trộn trọng số Model Soup (52% Thick + 48% Thin)...")
        sd_thick = _extract_sd(thick_found)
        sd_thin = _extract_sd(thin_found)
        blended_sd = {}
        for k in base_model.state_dict().keys():
            if k in sd_thick and k in sd_thin and sd_thick[k].shape == sd_thin[k].shape:
                blended_sd[k] = 0.52 * sd_thick[k].float() + 0.48 * sd_thin[k].float()
            elif k in sd_thick:
                blended_sd[k] = sd_thick[k]
            elif k in sd_thin:
                blended_sd[k] = sd_thin[k]
        base_model.load_state_dict(blended_sd, strict=False)
        print(f"[🌟 DUAL-CHECKPOINT MODEL SOUP]: Đã nạp thành công trọng số pha trộn! Khởi đầu lý tưởng Density ~ 1.02.")
        loaded_ckpt = True
        del sd_thick, sd_thin, blended_sd
    except Exception as e:
        print(f"[!] Pha trộn Model Soup gặp lỗi: {e}. Chuyển sang nạp đơn lẻ.")

# Cơ chế 2: Nạp đơn lẻ ưu tiên Thick > Thin > Fallback
if not loaded_ckpt:
    single_list = [p for p in [thick_found, thin_found] if p is not None] + thick_candidates + thin_candidates + fallback_candidates
    for cp in single_list:
        if os.path.exists(cp):
            try:
                print(f"[*] Đang nạp checkpoint đơn lẻ: {cp}...")
                sd = _extract_sd(cp)
                base_model.load_state_dict(sd, strict=False)
                print(f"[✓] Đã nạp thành công trọng số nền tảng (strict=False) từ: {cp}")
                loaded_ckpt = True
                del sd
                break
            except Exception as e:
                print(f"[!] Thử nạp {cp} gặp lỗi: {e}")

if not loaded_ckpt and os.path.exists(KAGGLE_INPUT):
    print("[*] Không tìm thấy ở đường dẫn chuẩn, đang quét nhanh (shallow scan) trong /kaggle/input...")
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        dirs[:] = [d for d in dirs if d.lower() not in {'frames', 'frame0', 'frame1', 'frame_gt', 'matches', 'images', 'clips', 'triplets', 'data'}]
        depth = os.path.relpath(root, KAGGLE_INPUT).count(os.sep)
        if depth >= 2:
            dirs.clear()
        for f in files:
            if f.endswith('.pt') and any(k in f.lower() for k in ['tps', 'baseline', 'checkpoint', 'model']):
                candidate_p = os.path.join(root, f)
                try:
                    sd = _extract_sd(candidate_p)
                    base_model.load_state_dict(sd, strict=False)
                    print(f"[✓] Đã tìm thấy và nạp thành công trọng số nền tảng từ: {candidate_p}")
                    loaded_ckpt = True
                    del sd
                    break
                except Exception as e:
                    print(f"[!] Thử nạp {candidate_p} gặp lỗi: {e}")
        if loaded_ckpt:
            break

if not loaded_ckpt:
    print("[*] Huấn luyện từ trạng thái ngẫu nhiên (Scratch Initialization).")

cleanup_memory()

trainable_params = [p for p in base_model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW([
    {'params': list(base_model.encoder.parameters()) + list(base_model.IFNet.parameters()), 'lr': 1.2e-5},
    {'params': list(base_model.asf_head.parameters()) + list(base_model.unet.parameters()), 'lr': 4.0e-5}
], betas=(0.9, 0.999), weight_decay=1e-4)

TOTAL_EPOCHS = 25
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=TOTAL_EPOCHS,
    eta_min=1.0e-6
)

scaler = torch.amp.GradScaler('cuda')

class ParallelTrainingWrapper_v1(nn.Module):
    def __init__(self, model, criterion):
        super(ParallelTrainingWrapper_v1, self).__init__()
        self.base_model = model
        self.loss_engine = criterion

    def forward(self, x0, tgt, x1, t_vals, c0_b, th01_b, c1_b, th10_b, valid_b, compute_loss=True):
        with torch.amp.autocast('cuda'):
            pred = self.base_model(
                x0, x1, t_vals,
                tps_precomputed=(c0_b, th01_b, c1_b, th10_b, valid_b)
            )
            if not compute_loss or tgt is None:
                return pred.detach()
            total_loss, l_l1, l_dice, l_density, l_fine = self.loss_engine(pred, tgt)
            total_out = total_loss.unsqueeze(0) if self.training else total_loss.detach().unsqueeze(0)
            return (
                pred.detach(),
                total_out,
                l_l1.detach().unsqueeze(0),
                l_dice.detach().unsqueeze(0),
                l_density.detach().unsqueeze(0),
                l_fine.detach().unsqueeze(0)
            )

raw_training_wrapper = ParallelTrainingWrapper_v1(base_model, loss_engine)

if NUM_GPUS > 1:
    print(f"🚀 KÍCH HOẠT PyTorch DataParallel trên {NUM_GPUS} GPUs (Forward + Full Loss Engine song song).")
    parallel_engine = nn.DataParallel(raw_training_wrapper)
else:
    parallel_engine = raw_training_wrapper

print(f"[*] Tổng số tham số huấn luyện V2.5: {sum(p.numel() for p in trainable_params):,}")
print(f"    - Nhóm 1 (Backbone ContextNet + IFNet Flow) : lr = {optimizer.param_groups[0]['lr']:.2e}")
print(f"    - Nhóm 2 (ASF-GR Head + Synthesis UNet)     : lr = {optimizer.param_groups[1]['lr']:.2e}")

In [ ]:
# [CELL 8]: Hàm Đo Lường Chỉ Số Thẩm Định Chống Bẫy V2.5 (100% Pure GPU Accelerated Metrics)
def _create_gaussian_kernel2d(kernel_size=11, sigma=1.5, device='cuda'):
    coords = torch.arange(kernel_size, dtype=torch.float32, device=device) - (kernel_size - 1) / 2.0
    g = torch.exp(-(coords ** 2) / (2.0 * sigma ** 2))
    g = g / g.sum()
    kernel_2d = (g.unsqueeze(1) @ g.unsqueeze(0)).unsqueeze(0).unsqueeze(0)
    return kernel_2d

_GAUSSIAN_KERNEL = None

def calculate_ssim_gpu(pred_1c, gt_1c):
    """Tính SSIM trực tiếp trên GPU bằng conv2d trong <1ms, 0 byte CPU RAM."""
    global _GAUSSIAN_KERNEL
    dev = pred_1c.device
    if _GAUSSIAN_KERNEL is None or _GAUSSIAN_KERNEL.device != dev:
        _GAUSSIAN_KERNEL = _create_gaussian_kernel2d(kernel_size=11, sigma=1.5, device=dev)

    pad = 5
    C1 = (0.01) ** 2
    C2 = (0.03) ** 2

    x = 1.0 - pred_1c
    y = 1.0 - gt_1c

    mu1 = F.conv2d(x, _GAUSSIAN_KERNEL, padding=pad)
    mu2 = F.conv2d(y, _GAUSSIAN_KERNEL, padding=pad)
    mu1_sq, mu2_sq = mu1 ** 2, mu2 ** 2
    mu1_mu2 = mu1 * mu2

    sig1_sq = F.conv2d(x * x, _GAUSSIAN_KERNEL, padding=pad) - mu1_sq
    sig2_sq = F.conv2d(y * y, _GAUSSIAN_KERNEL, padding=pad) - mu2_sq
    sig12 = F.conv2d(x * y, _GAUSSIAN_KERNEL, padding=pad) - mu1_mu2

    ssim_map = ((2.0 * mu1_mu2 + C1) * (2.0 * sig12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sig1_sq + sig2_sq + C2) + 1e-8)
    return float(torch.clamp(ssim_map.mean(), 0.0, 1.0).item())

def calculate_metrics_v1(pred_tensor, gt_tensor):
    """
    Tính toàn bộ metrics trên GPU (PSNR, SSIM, Stroke Recall, Ghost Pixels).
    Zero CPU memory transfer, zero OpenCV, zero glibc heap allocation!
    """
    with torch.no_grad():
        # 1. PSNR trên GPU
        mse = F.mse_loss(pred_tensor, gt_tensor).item()
        psnr = float(20.0 * math.log10(1.0 / math.sqrt(max(mse, 1e-10))))

        # 2. SSIM trên GPU
        ssim = calculate_ssim_gpu(pred_tensor, gt_tensor)

        # 3. Stroke Recall & F1-Stroke trên GPU
        p_mask = (pred_tensor > 0.15).float()
        g_mask = (gt_tensor > 0.15).float()
        p_lines = int(torch.sum(p_mask).item())
        g_lines = int(torch.sum(g_mask).item())

        if g_lines < 50:
            recall = 1.0
            precision = 1.0
            f1_stroke = 1.0
            p_dilated = p_mask
            g_dilated = g_mask
        else:
            p_dilated = F.max_pool2d(p_mask, kernel_size=7, stride=1, padding=3)
            hit = torch.sum((g_mask > 0) & (p_dilated > 0)).item()
            recall = float(hit / max(1, g_lines))

            # Dilated Stroke Precision & F1-Score: Đo độ sắc gọn và đầy đủ của nét vẽ
            g_dilated = F.max_pool2d(g_mask, kernel_size=7, stride=1, padding=3)
            prec_hit = torch.sum((p_mask > 0) & (g_dilated > 0)).item()
            precision = float(prec_hit / max(1, p_lines)) if p_lines > 0 else 0.0
            f1_stroke = float(2.0 * precision * recall / max(1e-6, precision + recall))

        is_blank = bool(p_lines < 100 and g_lines >= 500)

        # 4. Ghost Pixels & Density Ratio Chuẩn Hóa V2.5 (Bit-exact với Fast Benchmark: Không dilation mask)
        pred_u8 = ((1.0 - pred_tensor) * 255.0).clamp(0, 255).to(torch.uint8)
        gt_u8 = ((1.0 - gt_tensor) * 255.0).clamp(0, 255).to(torch.uint8)
        ghost_mask = (pred_u8 >= 180) & (pred_u8 <= 250) & (gt_u8 > 250)
        ghost_pixels = int(torch.sum(ghost_mask).item()) if not is_blank else -1
        density_ratio = float(p_lines / max(1, g_lines))

        # 5. Tính nhanh Chamfer Distance trên GPU từ p_dilated và g_dilated có sẵn (Zero extra pooling)
        chamfer_t = torch.mean(p_mask * (1.0 - g_dilated)) + torch.mean(g_mask * (1.0 - p_dilated))
        chamfer_distance = float(chamfer_t.item() * 0.5)

    return {
        'psnr': psnr,
        'ssim': ssim,
        'recall': recall,
        'precision': precision,
        'f1_stroke': f1_stroke,
        'ghost_pixels': ghost_pixels,
        'density_ratio': density_ratio,
        'chamfer_distance': chamfer_distance,
        'p_lines': p_lines,
        'g_lines': g_lines,
        'is_blank': is_blank
    }

def calculate_metrics_batch_gpu(pred_tensor, gt_tensor):
    """
    [V2.5 SIÊU TỐC]: Tính toàn bộ metrics trên GPU cho cả batch (B, 1, H, W) hoàn toàn song song.
    Chỉ đồng bộ CPU 1 lần duy nhất trên toàn batch (loại bỏ hơn 8,000 lần gọi .item() gây nghẽn PCIe bus).
    Bit-exact 100% với calculate_metrics_v1.
    """
    global _GAUSSIAN_KERNEL
    dev = pred_tensor.device
    B = pred_tensor.shape[0]

    with torch.no_grad():
        # 1. Batched PSNR
        mse_vec = torch.mean((pred_tensor - gt_tensor) ** 2, dim=[-3, -2, -1])
        psnr_vec = 20.0 * torch.log10(1.0 / torch.sqrt(torch.clamp(mse_vec, min=1e-10)))

        # 2. Batched SSIM
        if _GAUSSIAN_KERNEL is None or _GAUSSIAN_KERNEL.device != dev:
            _GAUSSIAN_KERNEL = _create_gaussian_kernel2d(kernel_size=11, sigma=1.5, device=dev)
        pad = 5
        C1 = (0.01) ** 2
        C2 = (0.03) ** 2
        x = 1.0 - pred_tensor
        y = 1.0 - gt_tensor
        mu1 = F.conv2d(x, _GAUSSIAN_KERNEL, padding=pad)
        mu2 = F.conv2d(y, _GAUSSIAN_KERNEL, padding=pad)
        mu1_sq, mu2_sq = mu1 ** 2, mu2 ** 2
        mu1_mu2 = mu1 * mu2
        sig1_sq = F.conv2d(x * x, _GAUSSIAN_KERNEL, padding=pad) - mu1_sq
        sig2_sq = F.conv2d(y * y, _GAUSSIAN_KERNEL, padding=pad) - mu2_sq
        sig12 = F.conv2d(x * y, _GAUSSIAN_KERNEL, padding=pad) - mu1_mu2
        ssim_map = ((2.0 * mu1_mu2 + C1) * (2.0 * sig12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sig1_sq + sig2_sq + C2) + 1e-8)
        ssim_vec = torch.clamp(torch.mean(ssim_map, dim=[-3, -2, -1]), 0.0, 1.0)

        # 3. Batched Stroke Recall, Precision, F1
        p_mask = (pred_tensor > 0.15).float()
        g_mask = (gt_tensor > 0.15).float()
        p_lines = torch.sum(p_mask, dim=[-3, -2, -1])
        g_lines = torch.sum(g_mask, dim=[-3, -2, -1])

        p_dilated = F.max_pool2d(p_mask, kernel_size=7, stride=1, padding=3)
        g_dilated = F.max_pool2d(g_mask, kernel_size=7, stride=1, padding=3)

        hit = torch.sum((g_mask > 0) & (p_dilated > 0), dim=[-3, -2, -1])
        prec_hit = torch.sum((p_mask > 0) & (g_dilated > 0), dim=[-3, -2, -1])

        sparse_gt = (g_lines < 50)
        recall_vec = torch.where(sparse_gt, torch.ones_like(p_lines), hit / torch.clamp(g_lines, min=1.0))
        prec_vec = torch.where(sparse_gt, torch.ones_like(p_lines), torch.where(p_lines > 0, prec_hit / torch.clamp(p_lines, min=1.0), torch.zeros_like(p_lines)))
        f1_vec = torch.where(sparse_gt, torch.ones_like(p_lines), 2.0 * prec_vec * recall_vec / torch.clamp(prec_vec + recall_vec, min=1e-6))

        # 4. Batched Ghost Pixels & Density Ratio
        pred_u8 = ((1.0 - pred_tensor) * 255.0).clamp(0, 255).to(torch.uint8)
        gt_u8 = ((1.0 - gt_tensor) * 255.0).clamp(0, 255).to(torch.uint8)
        ghost_mask = (pred_u8 >= 180) & (pred_u8 <= 250) & (gt_u8 > 250)
        ghost_px = torch.sum(ghost_mask.float(), dim=[-3, -2, -1])
        is_blank = (p_lines < 100) & (g_lines >= 500)
        ghost_vec = torch.where(is_blank, torch.full_like(ghost_px, -1.0), ghost_px)
        density_vec = p_lines / torch.clamp(g_lines, min=1.0)

        # 5. Batched Chamfer Distance
        g_dil_eff = torch.where(sparse_gt.view(B, 1, 1, 1), g_mask, g_dilated)
        p_dil_eff = torch.where(sparse_gt.view(B, 1, 1, 1), p_mask, p_dilated)
        chamfer_vec = 0.5 * (torch.mean(p_mask * (1.0 - g_dil_eff), dim=[-3, -2, -1]) + torch.mean(g_mask * (1.0 - p_dil_eff), dim=[-3, -2, -1]))

        # Chuyển về CPU duy nhất 1 lần cho cả batch
        stacked = torch.stack([
            psnr_vec, ssim_vec, recall_vec, prec_vec, f1_vec, ghost_vec, density_vec, chamfer_vec
        ], dim=1).cpu().numpy()

    return {
        'psnr': stacked[:, 0].tolist(),
        'ssim': stacked[:, 1].tolist(),
        'recall': stacked[:, 2].tolist(),
        'precision': stacked[:, 3].tolist(),
        'f1_stroke': stacked[:, 4].tolist(),
        'ghost_pixels': [int(g) for g in stacked[:, 5].tolist()],
        'density_ratio': stacked[:, 6].tolist(),
        'chamfer_distance': stacked[:, 7].tolist(),
    }

print("[✓] CELL 8 hoàn tất: Bộ chỉ số Anti-Cheat Metrics V2.5 (Kèm GPU Chamfer Siêu Tốc) (Kèm Dilated Stroke F1) 100% GPU sẵn sàng.")

In [ ]:
# [CELL 9]: Bộ Quản Lý Checkpoint Thông Minh V2.5 (Precision Hard Gates & Guaranteed Best Selection)
class SmartCheckpointManager_v25:
    def __init__(self, save_dir, ref_ghost=14322.38, ref_chamfer=0.002241, ref_recall=0.923635,
                 min_recall=0.895, min_density=0.90, max_density=1.10, target_density=1.00, gdrive_manager=None):
        self.save_dir = save_dir
        self.ref_ghost = ref_ghost
        self.ref_chamfer = ref_chamfer
        self.ref_recall = ref_recall
        self.min_recall = min_recall
        self.min_density = min_density
        self.max_density = max_density
        self.target_density = target_density
        self.best_score = float('inf')
        self.best_penalized_score = float('inf')
        self.best_epoch = 0
        self.best_metrics = {}
        self.has_strict_best = False
        self.gdrive = gdrive_manager
        os.makedirs(save_dir, exist_ok=True)

    def calculate_composite_score(self, ghost, chamfer, recall, density_ratio):
        term_ghost = 0.28 * (ghost / max(1.0, self.ref_ghost))
        term_chamfer = 0.22 * (chamfer / max(1e-6, self.ref_chamfer))
        denom_recall = max(1e-4, 1.0 - self.ref_recall)
        term_recall = 0.25 * (max(0.0, 1.0 - recall) / denom_recall)
        term_density = 0.25 * (abs(density_ratio - self.target_density) / 0.10)
        return float(term_ghost + term_chamfer + term_recall + term_density)

    def evaluate_and_save(self, model, current_epoch, metrics, total_epochs=None):
        recall = metrics['recall']
        psnr = metrics['psnr']
        ssim = metrics['ssim']
        density_ratio = metrics.get('density_ratio', 1.0)
        ghost = max(0, metrics.get('ghost_pixels', 0))
        chamfer = metrics.get('chamfer_distance', 0.002241)

        # 1. Hard Gates Chuẩn Sản Xuất V2.5
        gates_passed = True
        gate_reasons = []
        is_blank = metrics.get('is_blank', False)
        if is_blank:
            gates_passed = False
            gate_reasons.append("Blank Collapse")
        if recall < self.min_recall:
            gates_passed = False
            gate_reasons.append(f"Recall ({recall*100:.2f}%) < {self.min_recall*100:.1f}%")
        if density_ratio < self.min_density or density_ratio > self.max_density:
            gates_passed = False
            gate_reasons.append(f"Density ({density_ratio:.4f}) ngoài [{self.min_density:.2f}, {self.max_density:.2f}]")

        score = self.calculate_composite_score(ghost, chamfer, recall, density_ratio)

        raw_model = model
        if hasattr(raw_model, 'module'):
            raw_model = raw_model.module
        if hasattr(raw_model, 'base_model'):
            raw_model = raw_model.base_model
        cpu_state_dict = {k: v.detach().cpu() for k, v in raw_model.state_dict().items()}

        is_best = False
        is_progress = False
        best_path = os.path.join(self.save_dir, 'model_tps_v2.5_best.pt')
        progress_path = os.path.join(self.save_dir, 'model_tps_v2.5_best_progress.pt')
        penalized_score = score

        try:
            # Luôn cập nhật và lưu đè file latest
            latest_path = os.path.join(self.save_dir, 'model_tps_v2.5_latest.pt')
            torch.save(cpu_state_dict, latest_path)

            is_final_epoch = (total_epochs is not None and current_epoch >= total_epochs)
            if is_final_epoch:
                last_path = os.path.join(self.save_dir, 'model_tps_v2.5_last.pt')
                torch.save(cpu_state_dict, last_path)
                if self.gdrive:
                    self.gdrive.sync_file(last_path, 'model_tps_v2.5_last.pt')

            # Tầng 1: Đạt toàn bộ Hard Gates chính thức
            if gates_passed:
                self.has_strict_best = True
                if score < self.best_score:
                    self.best_score = score
                    self.best_epoch = current_epoch
                    self.best_metrics = dict(metrics)
                    self.best_metrics['composite_score'] = score
                    torch.save(cpu_state_dict, best_path)
                    if self.gdrive:
                        self.gdrive.sync_file(best_path, 'model_tps_v2.5_best.pt')
                        self.gdrive.sync_file(latest_path, 'model_tps_v2.5_latest.pt')
                    is_best = True

            # Tầng 2: Penalized Score Fallback Tracking
            gate_pen = 2.5 * max(0.0, self.min_recall - recall) + 2.0 * max(0.0, self.min_density - density_ratio) + 2.0 * max(0.0, density_ratio - self.max_density)
            penalized_score = score + gate_pen
            if not is_blank and penalized_score < self.best_penalized_score:
                self.best_penalized_score = penalized_score
                torch.save(cpu_state_dict, progress_path)
                if not self.has_strict_best:
                    torch.save(cpu_state_dict, best_path)
                if self.gdrive:
                    self.gdrive.sync_file(progress_path, 'model_tps_v2.5_best_progress.pt')
                is_progress = True
        finally:
            del cpu_state_dict
            gc.collect()

        return is_best, is_progress, score, penalized_score, gates_passed, gate_reasons, best_path, progress_path

# Tương thích ngược alias
SmartCheckpointManager_v24 = SmartCheckpointManager_v25
SmartCheckpointManager_v23 = SmartCheckpointManager_v25
SmartCheckpointManager_v22 = SmartCheckpointManager_v25

gdrive_syncer = None
ckpt_manager = SmartCheckpointManager_v25(
    save_dir=os.path.join(WORKING_DIR, 'checkpoints_v2.5'),
    gdrive_manager=None
)
print("[✓] CELL 9 hoàn tất: Smart Checkpoint Manager V2.5 (Precision Hard Gates & Guaranteed Best Selection) sẵn sàng.")

In [ ]:
# [CELL 10]: Vòng Lặp Huấn Luyện & Thẩm Định Hoàn Chỉnh (25 Epochs V2.5 - Exact 5-Item Output & Realtime Visual Demo)
GRAD_ACCUM_STEPS = 1
history = []
train_start_time = time.time()
history_csv_path = os.path.join(WORKING_DIR, 'training_history_v2.5.csv')

def generate_checkpoint_visual_demo(model, current_epoch, val_ds, dev, working_dir, num_samples=3):
    demo_dir = os.path.join(working_dir, 'visual_samples_v2.5')
    os.makedirs(demo_dir, exist_ok=True)

    step_idx = max(1, len(val_ds) // (num_samples + 1))
    sample_indices = [step_idx, step_idx * 2, min(len(val_ds) - 1, step_idx * 3)]

    raw_model = model
    if hasattr(raw_model, 'module'):
        raw_model = raw_model.module
    if hasattr(raw_model, 'base_model'):
        raw_model = raw_model.base_model
    was_training = raw_model.training
    raw_model.eval()

    saved_files = []
    with torch.inference_mode():
        for s_idx, idx in enumerate(sample_indices):
            batch_single = custom_collate_fn_v1([val_ds[idx]])
            t0_dev = batch_single[0].to(dev)
            tgt_dev = batch_single[1].to(dev)
            t1_dev = batch_single[2].to(dev)
            t_val_dev = batch_single[3].to(dev)
            c0_b = batch_single[4].to(dev)
            th01_b = batch_single[5].to(dev)
            c1_b = batch_single[6].to(dev)
            th10_b = batch_single[7].to(dev)
            valid_b = batch_single[8].to(dev)
            clip_name = str(batch_single[9][0])

            with torch.amp.autocast('cuda'):
                pred = raw_model(
                    t0_dev, t1_dev, t_val_dev,
                    tps_precomputed=(c0_b, th01_b, c1_b, th10_b, valid_b)
                )

            img_0 = ((1.0 - t0_dev) * 255.0).clamp(0, 255).byte().cpu().numpy()[0, 0]
            img_pred = ((1.0 - pred) * 255.0).clamp(0, 255).byte().cpu().numpy()[0, 0]
            img_gt = ((1.0 - tgt_dev) * 255.0).clamp(0, 255).byte().cpu().numpy()[0, 0]

            p_bin = (pred > 0.15).float().sum().item()
            g_bin = (tgt_dev > 0.15).float().sum().item()
            s_density = p_bin / max(1.0, g_bin)

            h, w = img_0.shape
            panel = np.hstack([img_0, img_pred, img_gt])
            cv2.putText(panel, "Keyframe 0 (t=0.0)", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)
            cv2.putText(panel, f"V2.5 Ep{current_epoch:02d} (Dens: {s_density:.2f})", (w + 20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 150, 0), 2)
            cv2.putText(panel, f"Ground Truth (t={float(t_val_dev.item()):.2f})", (2*w + 20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

            out_fn = f"sample_ep{current_epoch:02d}_{s_idx+1:02d}_{clip_name}.png"
            out_p = os.path.join(demo_dir, out_fn)
            cv2.imwrite(out_p, panel)
            saved_files.append(out_p)

            del t0_dev, tgt_dev, t1_dev, t_val_dev, c0_b, th01_b, c1_b, th10_b, valid_b, pred
            del img_0, img_pred, img_gt, panel

    if was_training:
        raw_model.train()
    return saved_files

print("=" * 85)
print(f"🚀 BẮT ĐẦU HUẤN LUYỆN TINH CHỈNH TPS-INBETWEEN V2.5 ({TOTAL_EPOCHS} EPOCHS)")
print(f"   - Tập Dữ Liệu       : Sakuga Golden Dataset ({num_train_samples:,} train + {num_val_samples:,} val clips)")
print(f"   - Công nghệ cốt lõi : Exact-STE Density Calibration + Bidirectional Stroke Width Control")
print(f"   - Tốc độ học đa tầng: Differential LR (Encoder: 1.2e-5, ASF/UNet: 4.0e-5)")
print(f"   - Phân tán Multi-GPU: Forward + Full Loss Engine chạy song song trên {max(1, NUM_GPUS)} GPU")
print(f"   - Thẩm định chuẩn   : val_loader_full ({VAL_BATCH_SIZE} samples/batch) tính 100% GPU batched metrics")
print(f"   - An toàn dữ liệu   : Tự động lưu Checkpoint Kaggle + Realtime Visual Demo on Best")
print(f"   - Host RAM ban đầu  : {format_host_ram()}")
print("=" * 85)

try:
    for epoch in range(1, TOTAL_EPOCHS + 1):
        epoch_start_time = time.time()

        # ----------------------------------------------------
        # PHASE 1: HUẤN LUYỆN (TRAINING)
        # ----------------------------------------------------
        parallel_engine.train()
        train_loss_accum = torch.tensor(0.0, device=device)
        train_l1_accum = torch.tensor(0.0, device=device)
        train_dice_accum = torch.tensor(0.0, device=device)
        train_density_accum = torch.tensor(0.0, device=device)
        train_fine_accum = torch.tensor(0.0, device=device)
        optimizer.zero_grad(set_to_none=True)

        train_pbar = tqdm(
            train_loader,
            desc=f"🏋️ [Ep {epoch:02d}/{TOTAL_EPOCHS:02d}] Train",
            unit="batch",
            leave=False
        )

        for step, batch in enumerate(train_pbar):
            t0, tgt, t1, t_vals, c0_b, th01_b, c1_b, th10_b, valid_b, clips = batch
            t0 = t0.to(device)
            tgt = tgt.to(device)
            t1 = t1.to(device)
            t_vals = t_vals.to(device)
            c0_b = c0_b.to(device)
            th01_b = th01_b.to(device)
            c1_b = c1_b.to(device)
            th10_b = th10_b.to(device)
            valid_b = valid_b.to(device)
            del batch, clips

            with torch.amp.autocast('cuda'):
                pred, total_loss_vec, loss_l1_vec, loss_dice_vec, loss_density_vec, loss_fine_vec = parallel_engine(
                    t0, tgt, t1, t_vals, c0_b, th01_b, c1_b, th10_b, valid_b, compute_loss=True
                )
                total_loss = total_loss_vec.mean()
                loss_l1 = loss_l1_vec.mean()
                loss_dice = loss_dice_vec.mean()
                loss_density = loss_density_vec.mean()
                loss_fine = loss_fine_vec.mean()
                loss_scaled = total_loss / GRAD_ACCUM_STEPS

            scaler.scale(loss_scaled).backward()

            if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(base_model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            train_loss_accum += total_loss.detach()
            train_l1_accum += loss_l1.detach()
            train_dice_accum += loss_dice.detach()
            train_density_accum += loss_density.detach()
            train_fine_accum += loss_fine.detach()

            if step % 20 == 0 or step == len(train_loader) - 1:
                train_pbar.set_postfix({
                    'Loss': f"{total_loss.item():.4f}",
                    'DensL': f"{loss_density.item():.3f}",
                    'FineL': f"{loss_fine.item():.3f}"
                })

        train_pbar.close()
        del train_pbar, t0, tgt, t1, t_vals, c0_b, th01_b, c1_b, th10_b, valid_b
        del pred, total_loss_vec, loss_l1_vec, loss_dice_vec, loss_density_vec, loss_fine_vec, total_loss, loss_l1, loss_dice, loss_density, loss_fine, loss_scaled

        scheduler.step()
        num_train_batches = max(1, len(train_loader))
        mean_train_loss = float((train_loss_accum / num_train_batches).item())
        mean_train_l1 = float((train_l1_accum / num_train_batches).item())
        mean_train_dice = float((train_dice_accum / num_train_batches).item())
        mean_train_dens_l = float((train_density_accum / num_train_batches).item())
        mean_train_fine_l = float((train_fine_accum / num_train_batches).item())

        cleanup_memory()

        # ----------------------------------------------------
        # PHASE 2: THẨM ĐỊNH (VALIDATION - 100% BATCHED GPU METRICS)
        # ----------------------------------------------------
        val_engine = raw_training_wrapper
        val_engine.eval()
        val_loss_accum = torch.tensor(0.0, device=device)
        val_l1_accum = torch.tensor(0.0, device=device)
        val_psnr_list, val_ssim_list, val_recall_list, val_f1_list, val_ghost_list = [], [], [], [], []
        val_density_list, val_chamfer_list = [], []

        active_val_loader = val_loader_full

        val_pbar = tqdm(
            active_val_loader,
            desc=f"🔍 [Ep {epoch:02d}/{TOTAL_EPOCHS:02d}] Val  ",
            unit="batch",
            leave=False
        )

        val_batches_run = 0

        with torch.no_grad():
            for val_step, batch in enumerate(val_pbar):
                val_batches_run += 1

                t0, tgt, t1, t_vals, c0_b, th01_b, c1_b, th10_b, valid_b, clips = batch
                t0 = t0.to(device)
                tgt = tgt.to(device)
                t1 = t1.to(device)
                t_vals = t_vals.to(device)
                c0_b = c0_b.to(device)
                th01_b = th01_b.to(device)
                c1_b = c1_b.to(device)
                th10_b = th10_b.to(device)
                valid_b = valid_b.to(device)
                del batch, clips

                with torch.amp.autocast('cuda'):
                    pred = val_engine(
                        t0, tgt, t1, t_vals, c0_b, th01_b, c1_b, th10_b, valid_b, compute_loss=False
                    )
                    v_l1 = F.l1_loss(pred, tgt)

                val_loss_accum += v_l1.detach()
                val_l1_accum += v_l1.detach()

                batch_m = calculate_metrics_batch_gpu(pred, tgt)
                val_psnr_list.extend(batch_m['psnr'])
                val_ssim_list.extend(batch_m['ssim'])
                val_recall_list.extend(batch_m['recall'])
                val_f1_list.extend(batch_m['f1_stroke'])
                val_density_list.extend(batch_m['density_ratio'])
                val_chamfer_list.extend(batch_m['chamfer_distance'])
                for g in batch_m['ghost_pixels']:
                    if g >= 0:
                        val_ghost_list.append(g)

                if (val_step + 1) % 100 == 0:
                    gc.collect()

                if val_step % 10 == 0 or val_step == len(active_val_loader) - 1:
                    val_pbar.set_postfix({
                        'ValL1': f"{v_l1.item():.4f}",
                        'F1': f"{val_f1_list[-1]*100:.1f}%" if val_f1_list else "...",
                        'PSNR': f"{val_psnr_list[-1]:.1f}dB" if val_psnr_list else "..."
                    })

        val_pbar.close()
        del val_pbar, t0, tgt, t1, t_vals, c0_b, th01_b, c1_b, th10_b, valid_b
        del pred, v_l1, val_engine

        num_val_batches = max(1, val_batches_run)
        mean_val_loss = float((val_loss_accum / num_val_batches).item())
        mean_val_l1 = float((val_l1_accum / num_val_batches).item())
        mean_val_psnr = float(np.mean(val_psnr_list)) if val_psnr_list else 18.0
        mean_val_ssim = float(np.mean(val_ssim_list)) if val_ssim_list else 0.89
        mean_val_recall = float(np.mean(val_recall_list)) if val_recall_list else 0.88
        mean_val_f1 = float(np.mean(val_f1_list)) if val_f1_list else 0.90
        mean_val_ghost = float(np.mean(val_ghost_list)) if val_ghost_list else 6500.0
        mean_val_density = float(np.mean(val_density_list)) if val_density_list else 1.00
        mean_val_chamfer = float(np.mean(val_chamfer_list)) if val_chamfer_list else 0.002

        epoch_time = time.time() - epoch_start_time
        lr_enc = optimizer.param_groups[0]['lr']
        lr_unet = optimizer.param_groups[-1]['lr']

        val_metrics = {
            'val_loss': mean_val_loss,
            'train_loss': mean_train_loss,
            'psnr': mean_val_psnr,
            'ssim': mean_val_ssim,
            'recall': mean_val_recall,
            'f1_stroke': mean_val_f1,
            'ghost_pixels': mean_val_ghost,
            'density_ratio': mean_val_density,
            'chamfer_distance': mean_val_chamfer
        }

        # Đánh giá & Lưu Checkpoint
        is_best_ep, is_progress_ep, comp_score, penalized_score, gates_passed, gate_reasons, best_path, progress_path = ckpt_manager.evaluate_and_save(
            base_model, epoch, val_metrics, total_epochs=TOTAL_EPOCHS
        )

        # =========================================================================
        # ĐỊNH DẠNG XUẤT THÔNG TIN EPOCH CHUẨN XÁC (ĐÚNG 5 MỤC THEO YÊU CẦU)
        # =========================================================================
        print(f"\n📌 ĐÃ XONG EPOCH {epoch:02d}/{TOTAL_EPOCHS:02d} (Thời gian: {epoch_time:.1f}s)")
        print(f"🔍 VAL Metrics: Val L1: {mean_val_l1:.4f} | PSNR: {mean_val_psnr:.2f}dB | SSIM: {mean_val_ssim:.4f} | Ghost Pixels: {mean_val_ghost:.0f} px | Density Ratio: {mean_val_density:.4f} | Chamfer: {mean_val_chamfer:.6f} | Recall: {mean_val_recall*100:.1f}% | Stroke F1: {mean_val_f1*100:.1f}%")
        print(f"🎯 Sakuga Composite Score V2.5: {comp_score:.4f} (Kỷ lục: {ckpt_manager.best_score:.4f})")
        if not gates_passed and gate_reasons:
            print(f"⚠️ [Hard Gates BỊ CHẶN]: {', '.join(gate_reasons)}")
        if is_best_ep:
            print(f"🌟 [TIẾN TRÌNH TỐT NHẤT V2.5 - KỶ LỤC CHÍNH THỨC]: Thỏa mãn toàn bộ Hard Gates! Lưu Best Checkpoint Epoch {epoch:02d} (Score: {comp_score:.4f}) -> {best_path}")
        elif is_progress_ep:
            print(f"📈 [TIẾN TRÌNH TỐT NHẤT V2.5]: Cập nhật Best Progress Epoch {epoch:02d} (Score: {comp_score:.4f}, Pen: {penalized_score:.4f}, Dens: {mean_val_density:.4f}, Rec: {mean_val_recall*100:.1f}%) -> {progress_path}")

        # TỰ ĐỘNG CHẠY DEMO TẠO ẢNH MẪU LƯU TRONG WORKING MỖI KHI CẬP NHẬT CHECKPOINT TỐT NHẤT
        if is_best_ep or is_progress_ep:
            saved_demo_samples = generate_checkpoint_visual_demo(base_model, epoch, val_dataset, device, working_dir=WORKING_DIR, num_samples=3)
            print(f"🖼️ [DEMO ẢNH MẪU CHECKPOINT TỐT NHẤT]: Đã lưu {len(saved_demo_samples)} ảnh mẫu kiểm tra trực quan tại: {os.path.join(WORKING_DIR, 'visual_samples_v2.5')}")

        rss_gb_now, cg_gb_now = get_host_ram_gb()
        history.append({
            'epoch': epoch,
            'train_loss': mean_train_loss,
            'val_loss': mean_val_loss,
            'train_l1': mean_train_l1,
            'val_l1': mean_val_l1,
            'val_psnr': mean_val_psnr,
            'val_ssim': mean_val_ssim,
            'stroke_recall': mean_val_recall,
            'stroke_f1': mean_val_f1,
            'ghost_pixels': mean_val_ghost,
            'density_ratio': mean_val_density,
            'chamfer_distance': mean_val_chamfer,
            'composite_score': comp_score,
            'is_best': is_best_ep,
            'lr_unet': lr_unet,
            'lr_encoder': lr_enc,
            'epoch_time_sec': round(epoch_time, 1),
            'host_ram_rss_gb': round(rss_gb_now, 3) if rss_gb_now is not None else None,
            'host_ram_cgroup_gb': round(cg_gb_now, 3) if cg_gb_now is not None else None
        })
        pd.DataFrame(history).to_csv(history_csv_path, index=False)

        if gdrive_syncer and epoch % 5 == 0:
            gdrive_syncer.sync_file(history_csv_path, 'training_history_v2.5.csv')

        del val_psnr_list, val_ssim_list, val_recall_list, val_f1_list, val_ghost_list, val_density_list, val_chamfer_list, val_metrics
        del train_loss_accum, train_l1_accum, train_dice_accum, train_density_accum, train_fine_accum
        del val_loss_accum, val_l1_accum
        cleanup_memory()
        try:
            import ctypes
            ctypes.CDLL('libc.so.6').malloc_trim(0)
        except Exception:
            pass

    final_latest_pt = os.path.join(WORKING_DIR, 'checkpoints_v2.5', 'model_tps_v2.5_latest.pt')
    final_last_pt = os.path.join(WORKING_DIR, 'checkpoints_v2.5', 'model_tps_v2.5_last.pt')
    if os.path.exists(final_latest_pt) and not os.path.exists(final_last_pt):
        shutil.copy2(final_latest_pt, final_last_pt)
    print(f"\n[✓] Hoàn thành toàn bộ {TOTAL_EPOCHS} Epochs! Checkpoint cuối cùng sẵn sàng tại: {final_last_pt}")

except KeyboardInterrupt:
    print("\n[!] Nhận tín hiệu dừng từ người dùng. Đang lưu trạng thái hiện tại...")
    pd.DataFrame(history).to_csv(history_csv_path, index=False)
    interrupted_path = os.path.join(WORKING_DIR, 'checkpoints_v2.5', 'model_tps_v2.5_interrupted.pt')
    os.makedirs(os.path.dirname(interrupted_path), exist_ok=True)
    _cpu_sd = {k: v.detach().cpu() for k, v in base_model.state_dict().items()}
    torch.save(_cpu_sd, interrupted_path)
    del _cpu_sd
    if gdrive_syncer:
        gdrive_syncer.sync_file(interrupted_path, 'model_tps_v2.5_interrupted.pt')
        gdrive_syncer.sync_file(history_csv_path, 'training_history_v2.5.csv')
    print(f"[✓] Đã lưu an toàn tại: {interrupted_path}")
    cleanup_memory()

In [ ]:
# [CELL 11]: Trực Quan Hóa Tiến Trình Hội Tụ & Chỉ Số Huấn Luyện Qua Các Epoch V2.5
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

def plot_training_metrics_convergence(history_csv=None, save_dir=None):
    if history_csv is None:
        history_csv = os.path.join(WORKING_DIR, 'training_history_v2.5.csv')
    if save_dir is None:
        save_dir = WORKING_DIR

    if not os.path.exists(history_csv):
        print(f"[!] Không tìm thấy file lịch sử {history_csv} để vẽ biểu đồ.")
        return

    df = pd.read_csv(history_csv)
    if len(df) == 0:
        print("[!] File lịch sử rỗng.")
        return

    epochs = df['epoch'].values

    fig, axes = plt.subplots(2, 3, figsize=(21, 12), dpi=150)
    fig.suptitle('TPS-INBETWEEN FINETUNE V2.5 - TRAINING DYNAMICS & METRIC CONVERGENCE DASHBOARD\n'
                 'Tập dữ liệu: Sakuga Golden Dataset (+ Patch 2) | Warm-start: Dual-Checkpoint Model Soup (Exact-STE & Bidirectional Stroke Control)',
                 fontsize=16, fontweight='bold', y=0.98)

    # 1. LOSS EVOLUTION
    ax1 = axes[0, 0]
    ax1.plot(epochs, df['train_loss'], label='Train Total Loss', color='#1f77b4', lw=2.2, marker='o', ms=4)
    if 'val_loss' in df.columns:
        ax1.plot(epochs, df['val_loss'], label='Val Total Loss', color='#ff7f0e', lw=2.2, marker='s', ms=4)
    if 'train_l1' in df.columns:
        ax1.plot(epochs, df['train_l1'], label='Train L1 Loss', color='#2ca02c', lw=1.5, ls='--')
    ax1.set_title('1. Loss Convergence (Total & L1)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss Value')
    ax1.legend(loc='upper right', frameon=True)
    ax1.grid(True, alpha=0.3)

    # 2. DENSITY RATIO & LINE THICKENING CONTROL (CORE V2.5)
    ax2 = axes[0, 1]
    d_col = 'density_ratio' if 'density_ratio' in df.columns else None
    if d_col:
        ax2.plot(epochs, df[d_col], label='Val Density Ratio', color='#9467bd', lw=2.5, marker='D', ms=5)
        ax2.axhspan(0.96, 1.03, color='#2ca02c', alpha=0.20, label='Target Dead-Zone [0.96, 1.03]')
        ax2.axhspan(0.90, 1.10, color='#ffbb78', alpha=0.15, label='Hard Gate [0.90, 1.10]')
        ax2.axhline(1.4580, color='#d62728', lw=1.8, ls=':', label='V2.4 Benchmark (1.458 - Bloated)')
        ax2.axhline(0.6795, color='#e377c2', lw=1.5, ls='-.', label='V2.3 Benchmark (0.680 - Too Thin)')
        ax2.axhline(1.0000, color='#2ca02c', lw=1.4, ls='-', label='Ideal V2.5 Target (1.000)')
    ax2.set_title('2. Line Thickness & Exact Density Control (V2.5)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Density Ratio (Pred Ink / GT Ink)')
    ax2.legend(loc='upper right', frameon=True, fontsize=9)
    ax2.grid(True, alpha=0.3)

    # 3. GHOST PIXELS SUPPRESSION
    ax3 = axes[0, 2]
    if 'ghost_pixels' in df.columns:
        ax3.plot(epochs, df['ghost_pixels'], label='Val Ghost Pixels', color='#d62728', lw=2.2, marker='^', ms=5)
        ax3.axhline(14322.0, color='#8c564b', lw=1.8, ls=':', label='V2.0.2 Ref (14,322 px)')
        if len(df) > 0 and df['ghost_pixels'].min() > 0:
            min_ghost = df['ghost_pixels'].min()
            min_ep = df.loc[df['ghost_pixels'].idxmin(), 'epoch']
            ax3.annotate(f'Min: {int(min_ghost):,} px (Ep {int(min_ep)})',
                         xy=(min_ep, min_ghost), xytext=(min_ep, min_ghost + 2000),
                         arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6),
                         fontweight='bold', fontsize=9)
    ax3.set_title('3. Ghost Pixels Reduction', fontsize=12, fontweight='bold')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Ghost Pixels (Count)')
    ax3.legend(loc='upper right', frameon=True)
    ax3.grid(True, alpha=0.3)

    # 4. FINE-STROKE PRESERVATION (RECALL & F1)
    ax4 = axes[1, 0]
    if 'stroke_recall' in df.columns:
        rec_pct = df['stroke_recall'] * 100.0 if df['stroke_recall'].max() <= 1.0 else df['stroke_recall']
        ax4.plot(epochs, rec_pct, label='Val Stroke Recall (%)', color='#2ca02c', lw=2.2, marker='o', ms=4)
        ax4.axhline(89.5, color='#d62728', lw=1.5, ls='--', label='V2.5 Hard Gate Min (89.5%)')
        ax4.axhline(92.36, color='#17becf', lw=1.5, ls=':', label='V2.0.2 Ref (92.36%)')
    if 'stroke_f1' in df.columns:
        f1_pct = df['stroke_f1'] * 100.0 if df['stroke_f1'].max() <= 1.0 else df['stroke_f1']
        ax4.plot(epochs, f1_pct, label='Stroke F1-Score (%)', color='#8c564b', lw=1.8, ls='-.')
    ax4.set_title('4. Fine-Stroke Preservation (Recall & F1)', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('Percentage (%)')
    ax4.legend(loc='lower right', frameon=True)
    ax4.grid(True, alpha=0.3)

    # 5. SAKUGA COMPOSITE SCORE & BEST CHECKPOINT
    ax5 = axes[1, 1]
    score_col = 'composite_score' if 'composite_score' in df.columns else None
    if score_col:
        ax5.plot(epochs, df[score_col], label='Sakuga Composite Score V2.5', color='#17becf', lw=2.2, marker='d', ms=5)
        best_idx = df[score_col].idxmin()
        best_ep = df.loc[best_idx, 'epoch']
        best_sc = df.loc[best_idx, score_col]
        ax5.scatter([best_ep], [best_sc], color='#d62728', s=160, zorder=5, marker='*', label=f'Best Checkpoint (Ep {int(best_ep)})')
        ax5.annotate(f'Best: {best_sc:.4f}\n(Epoch {int(best_ep)})',
                     xy=(best_ep, best_sc), xytext=(best_ep, best_sc + 0.15),
                     arrowprops=dict(facecolor='#d62728', shrink=0.05, width=1.5, headwidth=7),
                     fontweight='bold', fontsize=9, color='#d62728')
    else:
        ax5.plot(epochs, df['val_psnr'], label='Val PSNR (dB)', color='#e377c2', lw=2.0)
    ax5.set_title('5. Sakuga Composite Score V2.5 (Lower is Better)', fontsize=12, fontweight='bold')
    ax5.set_xlabel('Epoch')
    ax5.set_ylabel('Composite Score Value')
    ax5.legend(loc='upper right', frameon=True)
    ax5.grid(True, alpha=0.3)

    # 6. DIFFERENTIAL LEARNING RATE SCHEDULE
    ax6 = axes[1, 2]
    if 'lr_unet' in df.columns and 'lr_encoder' in df.columns:
        ax6.plot(epochs, df['lr_unet'], label='Heads / UNet LR (Max 4.0e-5)', color='#e377c2', lw=2.0)
        ax6.plot(epochs, df['lr_encoder'], label='Backbone / Flow LR (Max 1.2e-5)', color='#7f7f7f', lw=2.0, ls='--')
        ax6.set_yscale('log')
    elif 'epoch_time_sec' in df.columns:
        ax6.bar(epochs, df['epoch_time_sec'], color='#7f7f7f', alpha=0.6, label='Epoch Duration (s)')
    ax6.set_title('6. Differential LR Schedule (Cosine Decay)', fontsize=12, fontweight='bold')
    ax6.set_xlabel('Epoch')
    ax6.set_ylabel('Learning Rate (Log Scale)')
    ax6.legend(loc='upper right', frameon=True)
    ax6.grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    out_fig_path = os.path.join(save_dir, 'training_metrics_convergence_v25.png')
    plt.savefig(out_fig_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"[✓] Đã xuất biểu đồ trực quan hóa tiến trình hội tụ tại: {out_fig_path}")

plot_training_metrics_convergence()

In [ ]:
# [CELL 12]: Kiểm Toán Trực Quan Định Tính & Bằng Chứng Bảo Tồn Nét Mảnh V2.5
cleanup_memory()

visual_dir = os.path.join(WORKING_DIR, 'visual_samples_v2.5')
os.makedirs(visual_dir, exist_ok=True)

# Tải checkpoint tốt nhất vừa chọn được để xuất ảnh kiểm toán
best_ckpt_eval = os.path.join(WORKING_DIR, 'checkpoints_v2.5', 'model_tps_v2.5_best.pt')
if not os.path.exists(best_ckpt_eval):
    best_ckpt_eval = os.path.join(WORKING_DIR, 'checkpoints_v2.5', 'model_tps_v2.5_best_progress.pt')

if os.path.exists(best_ckpt_eval):
    try:
        sd = torch.load(best_ckpt_eval, map_location='cpu')
        base_model.load_state_dict(sd, strict=False)
        print(f"[✓] Đã nạp thành công Checkpoint phục vụ kiểm toán: {best_ckpt_eval}")
        del sd
    except Exception as e:
        print(f"[!] Lỗi nạp checkpoint cho audit: {e}")

base_model.eval()

print("[*] Đang xuất 12 ảnh kiểm toán trực quan mẫu chứng minh chuẩn hóa độ dày nét V2.5...")
sample_indices = np.linspace(0, len(val_dataset) - 1, min(12, len(val_dataset)), dtype=int)

with torch.inference_mode():
    for rank, idx in enumerate(sample_indices):
        batch_single = custom_collate_fn_v1([val_dataset[idx]])
        t0_dev = batch_single[0].to(device)
        tgt_dev = batch_single[1].to(device)
        t1_dev = batch_single[2].to(device)
        t_val_dev = batch_single[3].to(device)
        c0_b = batch_single[4].to(device)
        th01_b = batch_single[5].to(device)
        c1_b = batch_single[6].to(device)
        th10_b = batch_single[7].to(device)
        valid_b = batch_single[8].to(device)
        clip_name = batch_single[9][0]

        with torch.amp.autocast('cuda'):
            pred = base_model(
                t0_dev, t1_dev, t_val_dev,
                tps_precomputed=(c0_b, th01_b, c1_b, th10_b, valid_b)
            )

        img_0 = ((1.0 - t0_dev) * 255.0).clamp(0, 255).byte().cpu().numpy()[0, 0]
        img_pred = ((1.0 - pred) * 255.0).clamp(0, 255).byte().cpu().numpy()[0, 0]
        img_gt = ((1.0 - tgt_dev) * 255.0).clamp(0, 255).byte().cpu().numpy()[0, 0]

        p_bin = (pred > 0.15).float().sum().item()
        g_bin = (tgt_dev > 0.15).float().sum().item()
        s_density = p_bin / max(1.0, g_bin)

        h, w = img_0.shape
        panel = np.hstack([img_0, img_pred, img_gt])
        cv2.putText(panel, "Keyframe 0 (t=0.0)", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)
        cv2.putText(panel, f"Finetuned V2.5 (Dens: {s_density:.2f})", (w + 20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 150, 0), 2)
        cv2.putText(panel, f"Ground Truth (t={float(t_val_dev.item()):.2f})", (2*w + 20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)

        out_path = os.path.join(visual_dir, f"sample_{rank+1:02d}_{clip_name}_v25.png")
        cv2.imwrite(out_path, panel)

cleanup_memory()
print(f"[✓] Đã xuất thành công {len(sample_indices)} ảnh kiểm toán tại: {visual_dir}")

In [ ]:
# [CELL 13]: Đóng Gói Thành Phẩm Sản Phẩm V2.5 & Xuất Link Tải Trực Tiếp (Kèm GDrive Sync)
import os
import shutil
import pandas as pd

history_df = pd.DataFrame(history)
history_csv_path = os.path.join(WORKING_DIR, 'training_history_v2.5.csv')
history_df.to_csv(history_csv_path, index=False)
print(f"[*] Đã xuất lịch sử huấn luyện V2.5 tại: {history_csv_path}")

ckpt_dir = os.path.join(WORKING_DIR, 'checkpoints_v2.5')
best_ckpt_path = os.path.join(ckpt_dir, 'model_tps_v2.5_best.pt')
progress_ckpt_path = os.path.join(ckpt_dir, 'model_tps_v2.5_best_progress.pt')
latest_ckpt_path = os.path.join(ckpt_dir, 'model_tps_v2.5_latest.pt')
last_ckpt_path = os.path.join(ckpt_dir, 'model_tps_v2.5_last.pt')

if not os.path.exists(last_ckpt_path) and os.path.exists(latest_ckpt_path):
    shutil.copy2(latest_ckpt_path, last_ckpt_path)

if os.path.exists(best_ckpt_path):
    size_mb = os.path.getsize(best_ckpt_path) / (1024 * 1024)
    print(f"[*] Best Checkpoint V2.5 sẵn sàng: {best_ckpt_path} ({size_mb:.2f} MB)")
elif os.path.exists(progress_ckpt_path):
    size_mb = os.path.getsize(progress_ckpt_path) / (1024 * 1024)
    print(f"[*] Best Progress Checkpoint V2.5 sẵn sàng: {progress_ckpt_path} ({size_mb:.2f} MB)")
    shutil.copy2(progress_ckpt_path, best_ckpt_path)
    print(f"[*] Đã liên kết Best Progress sang {best_ckpt_path} phục vụ inference downstream.")
else:
    print(f"[!] Không tìm thấy best checkpoint, sao chép latest...")
    if os.path.exists(latest_ckpt_path):
        shutil.copy2(latest_ckpt_path, best_ckpt_path)

if os.path.exists(last_ckpt_path):
    size_mb = os.path.getsize(last_ckpt_path) / (1024 * 1024)
    print(f"[*] Final/Last Checkpoint V2.5 sẵn sàng: {last_ckpt_path} ({size_mb:.2f} MB)")

archive_path = os.path.join(WORKING_DIR, 'tps_finetune_v2.5_artifacts.zip')
shutil.make_archive(os.path.join(WORKING_DIR, 'tps_finetune_v2.5_artifacts'), 'zip', root_dir=WORKING_DIR, base_dir='checkpoints_v2.5')

if gdrive_syncer:
    gdrive_syncer.sync_file(history_csv_path, 'training_history_v2.5.csv')
    if os.path.exists(best_ckpt_path):
        gdrive_syncer.sync_file(best_ckpt_path, 'model_tps_v2.5_best.pt')
    if os.path.exists(progress_ckpt_path):
        gdrive_syncer.sync_file(progress_ckpt_path, 'model_tps_v2.5_best_progress.pt')
    if os.path.exists(last_ckpt_path):
        gdrive_syncer.sync_file(last_ckpt_path, 'model_tps_v2.5_last.pt')
    if os.path.exists(archive_path):
        gdrive_syncer.sync_file(archive_path, 'tps_finetune_v2.5_artifacts.zip')

print("\n" + "=" * 80)
print("🎉 HOÀN TẤT TOÀN BỘ QUY TRÌNH HUẤN LUYỆN TINH CHỈNH V2.5 (EXACT-STE DENSITY RELEASE)!")
print(f"   - File Checkpoint Tốt Nhất  : {best_ckpt_path}")
print(f"   - File Checkpoint Cuối Cùng : {last_ckpt_path}")
print(f"   - File Lịch Sử Huấn Luyện   : {history_csv_path}")
print(f"   - File Zip Tổng Hợp Tải Về  : {archive_path}")
print("=" * 80)

_temp_dirs_to_clean = []
for _d in (EXTRACT_DIR, TMP_DIR):
    try:
        _abs = os.path.abspath(_d)
        if _abs.startswith(os.path.abspath(WORKING_DIR) + os.sep) and os.path.exists(_abs):
            _temp_dirs_to_clean.append(_abs)
    except Exception:
        pass

for _d in _temp_dirs_to_clean:
    if os.path.exists(_d):
        shutil.rmtree(_d, ignore_errors=True)
        print(f"[*] Đã xóa thư mục dữ liệu tạm: {_d}")

cleanup_memory()
print(f"[*] Host RAM sau dọn dẹp -> {format_host_ram()}")